In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Design Generator with WAV Marker Integration

This script generates participant-specific experimental design CSVs for a Peripersonal 
Space (PPS) experiment using embedded WAV markers to ensure accurate timing.

The workflow:
1. Read breath hold markers directly from WAV files (odd IDs = inhale, even IDs = exhale)
2. Calculate required number of trials for each participant design
3. Create concatenated audio with enough markers for all trials
4. Generate counterbalanced design for each participant
5. Assign trial conditions to markers based on breath phase constraints
6. Produce final design files with accurate timestamps

Author: AI Assistant
"""

import os
import wave
import struct
import numpy as np
import pandas as pd
import random
import traceback
import json
import re
import math
from pathlib import Path
from pydub import AudioSegment
import io
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory for output files
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\PPS_Experiment_Module\ExperimentLog",
        'audio_output_dir': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\PPS_Experiment_Module\ExperimentAudio",
        # Input audio files with breath markers
        'breathing_intro': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav",
        'breathing_segment': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles\BoxBreathing_repetitionsegment_with_markers.wav",
        'breathing_outro': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles\BoxBreathing_outro.wav",
        # Stimulus files
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles",
    },
    'box_breathing': {
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Noise types for looming stimuli (can be any labels)
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# ----------------------------------------------------------------------
# Helper Functions for WAV Cue Point Processing
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    """
    Convert milliseconds to timestamp in format MM:SS.S.
    
    Args:
        ms: Milliseconds
        
    Returns:
        String: formatted timestamp
    """
    minutes = int(ms // 60000)
    seconds = (ms % 60000) / 1000.0
    return f"{minutes:02d}:{seconds:.1f}"

def read_cue_points_from_wav(wav_file):
    """
    Extract cue points directly from a WAV file.
    
    Args:
        wav_file: Path to WAV file
        
    Returns:
        List of dictionaries with cue point information
    """
    cue_points = []
    
    try:
        # First, get sample rate from WAV file
        with wave.open(wav_file, 'rb') as wav:
            sample_rate = wav.getframerate()
            
        # Read the audio file with pydub to get duration
        audio = AudioSegment.from_wav(wav_file)
        
        # Now read the cue points
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Read file size (and skip)
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find all chunks, looking for 'cue ' chunk
            cue_chunk_found = False
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_header == b'cue ':
                    cue_chunk_found = True
                    # Found cue chunk
                    num_cue_points = int.from_bytes(f.read(4), byteorder='little')
                    logging.info(f"Found {num_cue_points} cue points in WAV file {os.path.basename(wav_file)}")
                    
                    # Read all cue points
                    for i in range(num_cue_points):
                        # Each cue point is 24 bytes
                        cue_id = int.from_bytes(f.read(4), byteorder='little')
                        position = int.from_bytes(f.read(4), byteorder='little')
                        data_chunk_id = f.read(4)  # Should be 'data'
                        
                        # Skip the rest of the cue point data
                        f.read(12)
                        
                        # Calculate timestamp in milliseconds
                        position_ms = (position * 1000) / sample_rate
                        
                        # Determine breath phase from cue ID parity
                        breath_phase = "inhale" if cue_id % 2 == 1 else "exhale"
                        
                        cue_points.append({
                            'id': cue_id,
                            'position': position,
                            'position_ms': position_ms,
                            'timestamp': convert_ms_to_timestamp(position_ms),
                            'breathphase': breath_phase,
                            'sample_rate': sample_rate
                        })
                    break  # Found the cue chunk, exit loop
                else:
                    # Skip this chunk
                    f.seek(chunk_pos + 8 + chunk_size)
        
        if not cue_chunk_found:
            logging.warning(f"No cue chunk found in WAV file {os.path.basename(wav_file)}")
    
    except Exception as e:
        logging.error(f"Error reading cue points from {os.path.basename(wav_file)}: {e}")
        traceback.print_exc()
    
    # Sort markers by position
    cue_points.sort(key=lambda x: x['position'])
    
    if CONFIG['debug_mode']:
        if cue_points:
            logging.info(f"First cue point: {cue_points[0]['timestamp']} ({cue_points[0]['breathphase']})")
            logging.info(f"Last cue point: {cue_points[-1]['timestamp']} ({cue_points[-1]['breathphase']})")
        else:
            logging.warning(f"No cue points extracted from {os.path.basename(wav_file)}")
    
    return cue_points

def adjust_cue_points_for_concatenation(cue_points, offset_ms):
    """
    Adjust cue points when concatenating audio files.
    
    Args:
        cue_points: List of cue point dictionaries
        offset_ms: Offset in milliseconds to add to each cue point
        
    Returns:
        List of adjusted cue point dictionaries
    """
    adjusted_points = []
    
    for point in cue_points:
        new_point = point.copy()
        new_point['position_ms'] += offset_ms
        new_point['timestamp'] = convert_ms_to_timestamp(new_point['position_ms'])
        adjusted_points.append(new_point)
    
    return adjusted_points

def concatenate_wav_with_markers(output_path, *wav_files):
    """
    Concatenate multiple WAV files while preserving markers.
    
    Args:
        output_path: Path to save concatenated file
        *wav_files: Paths to WAV files to concatenate
    
    Returns:
        Tuple of (path to concatenated file, list of all adjusted cue points)
    """
    # Read each WAV file and extract cue points
    segments = []
    all_cue_points = []
    current_offset_ms = 0
    
    for wav_file in wav_files:
        # Skip files that don't exist
        if not os.path.exists(wav_file):
            logging.warning(f"Skipping non-existent file: {wav_file}")
            continue
            
        # Read audio
        audio = AudioSegment.from_wav(wav_file)
        segments.append(audio)
        
        # Extract and adjust cue points
        cue_points = read_cue_points_from_wav(wav_file)
        adjusted_points = adjust_cue_points_for_concatenation(cue_points, current_offset_ms)
        all_cue_points.extend(adjusted_points)
        
        # Update offset for next segment
        current_offset_ms += len(audio)
    
    # Concatenate audio segments
    if segments:
        concatenated = segments[0]
        for segment in segments[1:]:
            concatenated += segment
    else:
        raise ValueError("No valid audio segments to concatenate")
    
    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Save concatenated audio
    concatenated.export(output_path, format="wav")
    
    # Sort all cue points by position
    all_cue_points.sort(key=lambda x: x['position_ms'])
    
    return output_path, all_cue_points

def create_concatenated_audio(required_markers, config):
    """
    Create concatenated audio file with enough markers for the experiment.
    
    Args:
        required_markers: Number of markers required
        config: Configuration dictionary
        
    Returns:
        Tuple of (path to concatenated file, list of all markers)
    """
    # Check files exist
    intro_file = config['paths']['breathing_intro']
    segment_file = config['paths']['breathing_segment']
    outro_file = config['paths']['breathing_outro']
    
    if not os.path.exists(intro_file):
        raise FileNotFoundError(f"Intro file not found: {intro_file}")
    if not os.path.exists(segment_file):
        raise FileNotFoundError(f"Segment file not found: {segment_file}")
    
    # Read markers from files to determine how many segments we need
    intro_markers = read_cue_points_from_wav(intro_file)
    segment_markers = read_cue_points_from_wav(segment_file)
    
    markers_per_segment = len(segment_markers)
    intro_marker_count = len(intro_markers)
    
    if markers_per_segment == 0:
        raise ValueError(f"No markers found in segment file: {segment_file}")
    
    # Calculate how many segments we need
    remaining_markers = max(0, required_markers - intro_marker_count)
    segments_needed = math.ceil(remaining_markers / markers_per_segment)
    
    logging.info(f"Creating concatenated audio for {required_markers} markers:")
    logging.info(f"  Intro markers: {intro_marker_count}")
    logging.info(f"  Markers per segment: {markers_per_segment}")
    logging.info(f"  Segments needed: {segments_needed}")
    
    # Create list of files to concatenate
    files_to_concatenate = [intro_file]
    
    # Add repetition segments
    for i in range(segments_needed):
        files_to_concatenate.append(segment_file)
    
    # Add outro if it exists
    if os.path.exists(outro_file):
        files_to_concatenate.append(outro_file)
    
    # Create output path
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(
        config['paths']['audio_output_dir'],
        f"concatenated_breathing_{segments_needed}segments_{timestamp}.wav"
    )
    
    # Concatenate files
    concat_path, all_markers = concatenate_wav_with_markers(
        output_path, 
        *files_to_concatenate
    )
    
    logging.info(f"Created concatenated audio: {concat_path}")
    logging.info(f"Total markers: {len(all_markers)}")
    
    return concat_path, all_markers

def markers_to_dataframe(markers):
    """
    Convert list of marker dictionaries to DataFrame.
    
    Args:
        markers: List of marker dictionaries
        
    Returns:
        DataFrame with marker information
    """
    # Create dataframe with required columns
    df = pd.DataFrame(markers)
    
    # Rename columns to match expected format
    df = df.rename(columns={
        'timestamp': 'retentionslot_timestamp',
        'position_ms': 'milliseconds',
        'position': 'sample_index'
    })
    
    # Add description column
    df['description'] = df.apply(
        lambda row: f"{row['breathphase'].capitalize()} hold marker", 
        axis=1
    )
    
    return df

# ----------------------------------------------------------------------
# CLASS: PPSDesignGeneratorWithMarkers
# ----------------------------------------------------------------------
class PPSDesignGeneratorWithMarkers:
    """Generates participant-specific CSV design files for PPS experiments using WAV markers."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directories
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        self.audio_output_dir = self.config['paths']['audio_output_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        os.makedirs(self.audio_output_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Set seed for reproducibility
        self.base_seed = 42
        random.seed(self.base_seed)
        np.random.seed(self.base_seed)
        
        # Create concatenated audio with markers (if not already exists)
        self._prepare_audio_and_markers()
        
        if self.config['debug_mode']:
            total_trials = sum(self.trial_counts.values())
            logging.info("Initialized PPSDesignGeneratorWithMarkers:")
            logging.info(f"  - Repetitions: {self.config['repetitions']}")
            logging.info(f"  - Practice trials: {self.config['practice_trials']}")
            logging.info(f"  - Conditions: {self.conditions}")
            logging.info(f"  - SOA values: {self.config['soa_conditions_ms']}")
            logging.info(f"  - Total trials per participant: {total_trials} + {self.config['practice_trials']} practice = {total_trials + self.config['practice_trials']}")
            logging.info(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """
        Calculate the number of trials for each condition, using the repetitions parameter
        to determine how many trials per SOA per condition.
        """
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            logging.info("Trial count calculation:")
            for cond, count in trial_counts.items():
                logging.info(f"  {cond}: {count} trials")
            logging.info(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _prepare_audio_and_markers(self):
        """
        Prepare concatenated audio file and extract markers.
        If a suitable file already exists, use it instead of creating a new one.
        """
        # Calculate total markers needed for any participant
        total_trials = sum(self.trial_counts.values())
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = total_trials + practice_trials
        
        # Look for existing concatenated audio files
        audio_dir = self.config['paths']['audio_output_dir']
        existing_files = [f for f in os.listdir(audio_dir) if f.startswith("concatenated_breathing_")]
        
        # Use existing file if it has enough markers
        self.concatenated_audio_path = None
        self.markers = None
        
        if existing_files:
            for file in sorted(existing_files, reverse=True):  # Try newest first
                try:
                    file_path = os.path.join(audio_dir, file)
                    markers = read_cue_points_from_wav(file_path)
                    
                    if len(markers) >= required_markers:
                        self.concatenated_audio_path = file_path
                        self.markers = markers
                        logging.info(f"Using existing audio file: {file}")
                        logging.info(f"  - Contains {len(markers)} markers, need {required_markers}")
                        break
                except Exception as e:
                    logging.warning(f"Error reading markers from {file}: {e}")
        
        # Create new file if needed
        if self.concatenated_audio_path is None:
            logging.info(f"No suitable existing audio file found. Creating new one...")
            self.concatenated_audio_path, self.markers = create_concatenated_audio(
                required_markers,
                self.config
            )
        
        # Convert markers to DataFrame for easier processing
        self.markers_df = markers_to_dataframe(self.markers)
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a counterbalanced design for a participant with trials per SOA
        determined by the repetitions parameter in the config.
        
        Catch trials don't have SOA values assigned since they don't have tactile stimuli.
        """
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = list(LOOMING_STIMULI.keys())
        repetitions = self.config['repetitions']  # Trials per SOA per condition
        
        all_trials = []
        
        # Generate main conditions: inhalation, exhalation, and baseline
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate exactly 'repetitions' trials
            for soa in soa_conditions:
                # Calculate how many trials per stimulus type (as evenly as possible)
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)  # Integer division
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute trials across stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    # Add extra trial for remainder distribution
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle the stimulus types
                random.shuffle(stim_trials)
                
                # Create trials for this SOA
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True  # For these main trials, tactile is present.
                    })
        
        # Add catch trials (looming only, no tactile) - no SOA values needed
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly across stimulus types only
            catch_trials = []
            stim_counts = {stim: 0 for stim in stimulus_types}
            
            for _ in range(catch_count):
                # Choose the stimulus type with the lowest count
                min_stim = min(stim_counts, key=stim_counts.get)
                
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,  # No SOA for catch trials
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                
                stim_counts[min_stim] += 1
            
            # Shuffle catch trials and add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle all trials and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        if self.config['debug_mode']:
            total_trials = len(all_trials_df)
            logging.info("\nTrial distribution:")
            logging.info(f"Total trials: {total_trials}")
            
            # Count by trial type
            for trial_type in sorted(all_trials_df['trial_type'].unique()):
                type_df = all_trials_df[all_trials_df['trial_type'] == trial_type]
                logging.info(f"Total {trial_type}: {len(type_df)}")
        
        return all_trials_df

    def assign_trials_to_markers(self, design_df, participant_id):
        """
        Assign trials to markers based on breath phase requirements.
        
        Key requirements:
        1. Inhalation trials must go to inhale breath phase markers
        2. Exhalation trials must go to exhale breath phase markers
        3. Baseline and catch trials distributed between inhale and exhale
        4. Practice trials at the beginning
        """
        # Account for practice trials in required markers calculation
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = len(design_df) + practice_trials
        
        # Ensure we have enough markers
        if len(self.markers) < required_markers:
            raise ValueError(f"Not enough markers ({len(self.markers)}) for required trials ({required_markers})")
        
        # Get copy of markers dataframe
        markers_df = self.markers_df.copy()
        
        # Ensure markers sorted by position
        markers_df = markers_df.sort_values(by='milliseconds').reset_index(drop=True)
        
        # Reserve the first N markers for practice trials
        practice_markers = markers_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_markers = markers_df.iloc[practice_trials:practice_trials+len(design_df)].copy()
        
        # Create practice trials
        practice_trial_list = []
        if practice_trials > 0:
            for i, (_, row) in enumerate(practice_markers.iterrows()):
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,  # Start numbering from 1
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),
                    'is_tactile': False,
                    'retentionslot_timestamp': row['retentionslot_timestamp'],
                    'milliseconds': row['milliseconds'],
                    'sample_index': row['sample_index'],
                    'description': row['description'],
                    'breathphase': row['breathphase']
                }
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by trial type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count markers by breath phase
        inhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'inhale']
        exhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'exhale']
        n_inhale_markers = len(inhale_markers)
        n_exhale_markers = len(exhale_markers)
        
        if self.config['debug_mode']:
            logging.info(f"Available markers: {n_inhale_markers} inhale, {n_exhale_markers} exhale")
            logging.info(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, " 
                      f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough markers for mandatory trials
        if len(pool_inh) > n_inhale_markers:
            raise ValueError(f"Not enough inhale markers ({n_inhale_markers}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale_markers:
            raise ValueError(f"Not enough exhale markers ({n_exhale_markers}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining markers per breath phase
        remaining_inhale = n_inhale_markers - len(pool_inh)
        remaining_exhale = n_exhale_markers - len(pool_exh)
        
        # Total baseline+catch trials
        total_flex_trials = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough markers for all trials
        if remaining_inhale + remaining_exhale < total_flex_trials:
            raise ValueError(f"Not enough remaining markers ({remaining_inhale + remaining_exhale}) "
                            f"for flexible trials ({total_flex_trials})")
        
        # Ensure optimal distribution of baseline and catch trials between breath phases
        
        # Shuffle baseline and catch trials
        random.shuffle(pool_baseline)
        random.shuffle(pool_catch)
        
        # Calculate how many of each should go to each breath phase
        baseline_ratio = remaining_inhale / (remaining_inhale + remaining_exhale)
        catch_ratio = baseline_ratio  # Same ratio for consistency
        
        baseline_inhale = min(int(len(pool_baseline) * baseline_ratio), remaining_inhale)
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = min(int(len(pool_catch) * catch_ratio), remaining_inhale - baseline_inhale)
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create inhale and exhale flexible trial pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        if self.config['debug_mode']:
            logging.info(f"Distributing flexible trials:")
            logging.info(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
            logging.info(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Create final allocation pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        # Shuffle within each pool to randomize order
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Create dictionaries mapping breath phase to trial pool and marker pool
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        phase_to_markers = {
            'inhale': inhale_markers.to_dict('records'),
            'exhale': exhale_markers.to_dict('records')
        }
        
        # Initialize counters for each breath phase
        pool_counters = {
            'inhale': 0,
            'exhale': 0
        }
        
        # Final list to store assigned trials
        regular_assignments = []
        
        # Assign trials to markers by breath phase
        for phase in ['inhale', 'exhale']:
            trials = phase_to_pool[phase]
            markers = phase_to_markers[phase]
            
            for i, trial in enumerate(trials):
                if i < len(markers):
                    marker = markers[i]
                    
                    # Combine trial information with marker information
                    combined = trial.copy()
                    for key, value in marker.items():
                        combined[key] = value
                    
                    regular_assignments.append(combined)
                else:
                    logging.warning(f"Not enough {phase} markers for all trials")
        
        # Sort assignments by marker position
        regular_assignments.sort(key=lambda x: x['milliseconds'])
        
        # Combine practice trials and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final dataframe and sort by marker position
        final_df = pd.DataFrame(final_assignments).sort_values(by='milliseconds').reset_index(drop=True)
        
        # Renumber trials from 1 to N to ensure sequential numbering
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Compute additional timing information
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        # Validate assignment
        self._validate_assignment(final_df)
        
        return final_df

    def _validate_assignment(self, assigned_df):
        """Validate that trial assignment respects breath phase requirements."""
        if not self.config['debug_mode']:
            return
            
        logging.info("\n=== ASSIGNMENT VALIDATION ===")
        
        # Skip practice trials for validation
        non_practice_df = assigned_df[assigned_df['trial_type'] != 'practice']
        
        # Check for inhalation trials assigned to inhale
        inh_trials = non_practice_df[non_practice_df['trial_type'] == 'inhalation']
        inh_correct = (inh_trials['breathphase'].str.lower() == 'inhale').all() if len(inh_trials) > 0 else True
        logging.info(f"Inhalation trials on inhale breathphase: {inh_correct}")
        
        # Check for exhalation trials assigned to exhale
        exh_trials = non_practice_df[non_practice_df['trial_type'] == 'exhalation']
        exh_correct = (exh_trials['breathphase'].str.lower() == 'exhale').all() if len(exh_trials) > 0 else True
        logging.info(f"Exhalation trials on exhale breathphase: {exh_correct}")
        
        # Count distribution of baseline and catch trials across breathphases
        baseline_inhale = ((non_practice_df['trial_type'] == 'baseline') & 
                         (non_practice_df['breathphase'].str.lower() == 'inhale')).sum()
        baseline_exhale = ((non_practice_df['trial_type'] == 'baseline') & 
                          (non_practice_df['breathphase'].str.lower() == 'exhale')).sum()
        catch_inhale = ((non_practice_df['trial_type'] == 'catch') & 
                       (non_practice_df['breathphase'].str.lower() == 'inhale')).sum()
        catch_exhale = ((non_practice_df['trial_type'] == 'catch') & 
                       (non_practice_df['breathphase'].str.lower() == 'exhale')).sum()
        
        logging.info(f"Baseline trials: {baseline_inhale} on inhale, {baseline_exhale} on exhale")
        logging.info(f"Catch trials: {catch_inhale} on inhale, {catch_exhale} on exhale")
        
        # Check trial sequence
        time_sorted = assigned_df.sort_values('milliseconds')
        prev_time = None
        timestamp_errors = 0
        
        for idx, row in time_sorted.iterrows():
            cur_time = row['milliseconds']
            if prev_time is not None and cur_time < prev_time:
                timestamp_errors += 1
                logging.warning(f"Timestamp error at index {idx}: {cur_time} < {prev_time}")
            prev_time = cur_time
        
        logging.info(f"Timestamp sequence errors: {timestamp_errors}")
        
        # Count practice trials
        practice_count = (assigned_df['trial_type'] == 'practice').sum()
        logging.info(f"Practice trials: {practice_count}")
    
    def finalize_design_csv(self, design_df):
        """
        Finalize the design CSV: Set appropriate null values and order columns.
        For baseline trials, remove looming stimulus info; for catch trials, remove tactile stimulus info.
        """
        # Set appropriate null values based on trial type
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Set expected_response based on trial type
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        # Determine column order
        marker_cols = ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        trial_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 
                      'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 
                      'soa_ms', 'jittered_ms', 'expected_response']
        
        # Combine columns in desired order
        all_cols = marker_cols + [col for col in trial_cols if col not in marker_cols and col in design_df.columns]
        remaining_cols = [col for col in design_df.columns if col not in all_cols]
        final_order = all_cols + remaining_cols
        
        # Reorder columns
        design_df = design_df[final_order]
        
        # Validate final design
        self._validate_design(design_df)
        
        return design_df
    
    def _validate_design(self, design_df):
        """Perform comprehensive validation and counterbalancing checks on the design."""
        if not self.config['debug_mode']:
            return
            
        logging.info("\n=== DESIGN VALIDATION ===")
        
        # Separate practice trials from experimental trials
        practice_df = design_df[design_df['trial_type'] == 'practice']
        exp_df = design_df[design_df['trial_type'] != 'practice']
        
        # 1. Overall trial count summary
        total_trials = len(design_df)
        practice_count = len(practice_df)
        experimental_count = len(exp_df)
        
        trial_counts = exp_df['trial_type'].value_counts().to_dict()
        
        logging.info(f"TOTAL TRIALS: {total_trials} (Practice: {practice_count}, Experimental: {experimental_count})")
        logging.info("Trial type distribution (excluding practice):")
        for trial_type, count in sorted(trial_counts.items()):
            percent = (count / experimental_count) * 100
            logging.info(f"  {trial_type.upper()}: {count} trials ({percent:.1f}%)")
        
        # 2. Detailed breakdown by trial type
        logging.info("\nStimulus distribution by trial type:")
        for trial_type in sorted(exp_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = exp_df[exp_df['trial_type'] == trial_type]
            stim_counts = subset['stimulus_type'].value_counts().to_dict()
            logging.info(f"  {trial_type.upper()}:")
            for stim, count in sorted(stim_counts.items()):
                if stim is not None:
                    percent = (count / len(subset)) * 100
                    logging.info(f"    {stim}: {count} trials ({percent:.1f}%)")
        
        # 3. SOA distribution checks
        logging.info("\nSOA distribution by trial type:")
        for trial_type in sorted(exp_df['trial_type'].unique()):
            if trial_type == 'baseline':
                continue
            subset = exp_df[exp_df['trial_type'] == trial_type]
            soa_counts = subset['soa_value_ms'].value_counts().to_dict()
            logging.info(f"  {trial_type.upper()}:")
            for soa, count in sorted(soa_counts.items()):
                if soa is not None:
                    percent = (count / len(subset)) * 100
                    logging.info(f"    {soa} ms: {count} trials ({percent:.1f}%)")
        
        # 4. Jitter distribution
        logging.info("\nJitter distribution (excluding practice):")
        jitter_counts = exp_df['jitter_ms'].value_counts().to_dict()
        for jitter, count in sorted(jitter_counts.items()):
            if jitter is not None:
                percent = (count / experimental_count) * 100
                logging.info(f"  {jitter} ms: {count} trials ({percent:.1f}%)")
        
        # 5. Breath phase checks for experimental trials
        logging.info("\nBreath phase distribution (excluding practice):")
        if 'breathphase' in exp_df.columns:
            phase_counts = exp_df['breathphase'].value_counts().to_dict()
            for phase, count in sorted(phase_counts.items()):
                percent = (count / experimental_count) * 100
                logging.info(f"  {phase}: {count} trials ({percent:.1f}%)")
        
        # 6. Check for timing sequence issues
        if 'milliseconds' in design_df.columns:
            time_sorted = design_df.sort_values('milliseconds')
            
            # Check for duplicated timestamps
            dup_timestamps = time_sorted['milliseconds'].duplicated().sum()
            if dup_timestamps > 0:
                logging.warning(f"Found {dup_timestamps} duplicated timestamps")
            
            # Check for proper ascending sequence
            prev_time = None
            seq_errors = 0
            
            for idx, row in time_sorted.iterrows():
                cur_time = row['milliseconds']
                if prev_time is not None and cur_time < prev_time:
                    seq_errors += 1
                prev_time = cur_time
            
            logging.info(f"Timestamp sequence errors: {seq_errors}")
        
        # 7. Check timing intervals
        if 'milliseconds' in design_df.columns:
            time_sorted = design_df.sort_values('milliseconds')
            intervals = []
            
            for i in range(1, len(time_sorted)):
                interval = time_sorted.iloc[i]['milliseconds'] - time_sorted.iloc[i-1]['milliseconds']
                intervals.append(interval)
            
            if intervals:
                min_interval = min(intervals)
                max_interval = max(intervals)
                avg_interval = sum(intervals) / len(intervals)
                
                logging.info(f"\nTiming intervals:")
                logging.info(f"  Minimum interval: {min_interval:.1f} ms")
                logging.info(f"  Maximum interval: {max_interval:.1f} ms")
                logging.info(f"  Average interval: {avg_interval:.1f} ms")
    
    def generate_participant_audio(self, participant_id, design_df):
        """
        Generate participant-specific audio files by adding stimuli to the concatenated audio.
        For now, this is a placeholder function that would be implemented in the audio generation module.
        """
        # Placeholder - this would be implemented in the actual audio generation module
        audio_output_paths = {
            'looming': os.path.join(self.audio_output_dir, f"participant_{participant_id}_looming.wav"),
            'tactile': os.path.join(self.audio_output_dir, f"participant_{participant_id}_tactile.wav"),
            'design': os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
        }
        
        # Save design file for audio generator
        design_df.to_csv(audio_output_paths['design'], index=False)
        
        logging.info(f"Participant {participant_id} design saved to {audio_output_paths['design']}")
        logging.info("Audio files would be generated from this design.")
        
        return audio_output_paths
    
    def process_participant(self, participant_id):
        """
        Process a single participant: generate design, assign to markers, finalize, and save CSV.
        Optionally generate participant-specific audio files.
        """
        try:
            if self.config['debug_mode']:
                logging.info(f"\n=== Processing participant {participant_id} ===")
            
            # Generate counterbalanced design
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # Assign trials to markers
            design_df = self.assign_trials_to_markers(design_df, participant_id)
            
            # Finalize design CSV
            design_df = self.finalize_design_csv(design_df)
            
            # Save design CSV
            out_path = os.path.join(self.experiment_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                logging.info(f"Saved participant {participant_id} design to {out_path}")
            
            # Generate participant audio (this would call the audio generator module)
            audio_paths = self.generate_participant_audio(participant_id, design_df)
            
            return True, {
                'design_path': out_path,
                'audio_paths': audio_paths
            }
            
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Run design generation for multiple participants.
        
        Args:
            participant_ids: List of specific participant IDs to process
            num_participants: Number of participants to process (if participant_ids not provided)
            
        Returns:
            Dictionary with results
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            logging.info("=== Starting PPSDesignGeneratorWithMarkers ===")
            logging.info(f"Repetitions: {self.config['repetitions']}")
            logging.info(f"Practice trials: {self.config['practice_trials']}")
            logging.info(f"Participants: {participant_ids}")
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        # Summary of all participants
        if self.config['debug_mode'] and len(results) > 1:
            logging.info("\n=== CROSS-PARTICIPANT VALIDATION ===")
            all_success = all(info['success'] for info in results.values())
            logging.info(f"All participants generated successfully: {all_success}")
        
        return results

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the design generator with WAV markers."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT DESIGN GENERATOR WITH WAV MARKERS =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials at beginning: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {list(LOOMING_STIMULI.keys())}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_base_trials + catch_trials + practice_trials}")
    
    # Ask for participant IDs to process
    while True:
        try:
            input_str = input("\nEnter participant IDs to process (comma-separated, or 'all' for all): ")
            if input_str.lower() == 'all':
                participant_ids = None
                break
            elif input_str.strip():
                participant_ids = [int(pid) for pid in input_str.split(',')]
                break
            else:
                print("Please enter participant IDs or 'all'")
        except ValueError:
            print("Invalid input. Please enter comma-separated numbers or 'all'")
    
    # Create the generator and run
    try:
        generator = PPSDesignGeneratorWithMarkers(CONFIG)
        results = generator.run(participant_ids=participant_ids)
        
        # Summarize results
        print("\n===== GENERATION SUMMARY =====")
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"Successfully generated designs for {success_count}/{len(results)} participants.")
        
        for pid, info in results.items():
            if info['success']:
                paths = info['paths']
                print(f"Participant {pid}: CSV generated at {paths['design_path']}")
            else:
                print(f"Participant {pid}: ERROR - no design generated.")
        
    except Exception as e:
        print(f"Error in design generation: {e}")
        traceback.print_exc()
    
    print("\nDesign generation complete. Audio files would be generated from these designs.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT DESIGN GENERATOR WITH WAV MARKERS =====
Repetitions (trials per SOA per condition): 12
Practice trials at beginning: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206



Enter participant IDs to process (comma-separated, or 'all' for all):  9


2025-03-27 19:44:05,591 - INFO - Trial count calculation:
2025-03-27 19:44:05,591 - INFO -   inhalation: 60 trials
2025-03-27 19:44:05,591 - INFO -   exhalation: 60 trials
2025-03-27 19:44:05,594 - INFO -   baseline: 60 trials
2025-03-27 19:44:05,594 - INFO -   catch: 18 trials
2025-03-27 19:44:05,594 - INFO -   Total: 198 trials
2025-03-27 19:44:05,631 - INFO - No suitable existing audio file found. Creating new one...


Error in design generation: Intro file not found: C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav

Design generation complete. Audio files would be generated from these designs.


Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3682072969.py", line 1096, in main
    generator = PPSDesignGeneratorWithMarkers(CONFIG)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3682072969.py", line 395, in __init__
    self._prepare_audio_and_markers()
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3682072969.py", line 470, in _prepare_audio_and_markers
    self.concatenated_audio_path, self.markers = create_concatenated_audio(
                                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3682072969.py", line 290, in create_concatenated_audio
    raise FileNotFoundError(f"Intro file not found: {intro_file}")
FileNotFoundError: Intro file not found: C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav


In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Simplified PPS Design Generator for Pre-Marked WAV Files

This script generates participant-specific experimental design CSVs for a Peripersonal 
Space (PPS) experiment using WAV files that already have markers embedded.

The workflow:
1. Concatenate existing WAV files that already have breath markers embedded
2. Extract all markers from the concatenated WAV file into a template CSV
3. Generate participant-specific design CSVs by:
   a. Creating counterbalanced trial conditions
   b. Assigning conditions to markers based on breathphase constraints
   c. Computing stimulus timing information
4. Save the concatenated WAV file and all CSVs in their designated locations
"""

import os
import wave
import struct
import numpy as np
import pandas as pd
import random
import traceback
import json
import re
import math
from pathlib import Path
from pydub import AudioSegment
import io
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory for output files
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'audio_output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        # Input audio files with breath markers
        'breathing_intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav",
        'breathing_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment_with_markers.wav",
        'breathing_outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
    },
    'box_breathing': {
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# Marker type constants
MARKER_TYPE_INHALATION = 1  # Odd IDs = Inhale
MARKER_TYPE_EXHALATION = 2  # Even IDs = Exhale

# ----------------------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    """Convert milliseconds to timestamp in format MM:SS.S."""
    minutes = int(ms // 60000)
    seconds = (ms % 60000) / 1000.0
    return f"{minutes:02d}:{seconds:.1f}"

def extract_markers_from_wav(wav_file):
    """Extract marker information directly from a WAV file's cue points."""
    markers = []
    
    try:
        # First get sample rate
        with wave.open(wav_file, 'rb') as wav:
            sample_rate = wav.getframerate()
        
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Skip file size
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find cue chunk
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_header == b'cue ':
                    # Found cue chunk
                    num_cue_points = int.from_bytes(f.read(4), byteorder='little')
                    logging.info(f"Found {num_cue_points} cue points in {os.path.basename(wav_file)}")
                    
                    # Read all cue points
                    for i in range(num_cue_points):
                        # Read cue ID
                        cue_id = int.from_bytes(f.read(4), byteorder='little')
                        # Read position
                        position = int.from_bytes(f.read(4), byteorder='little')
                        # Skip remaining cue point data (16 bytes)
                        f.read(16)
                        
                        # Calculate timestamp
                        position_ms = (position * 1000) / sample_rate
                        
                        # Determine breathphase from cue ID parity
                        breathphase = "inhale" if cue_id % 2 == 1 else "exhale"
                        
                        markers.append({
                            'id': cue_id,
                            'position': position,
                            'milliseconds': position_ms,
                            'timestamp': convert_ms_to_timestamp(position_ms),
                            'breathphase': breathphase,
                            'sample_rate': sample_rate
                        })
                    
                    break  # Exit loop after finding cue chunk
                else:
                    # Skip to next chunk
                    f.seek(chunk_pos + 8 + chunk_size)
    
    except Exception as e:
        logging.error(f"Error reading markers from {os.path.basename(wav_file)}: {e}")
        traceback.print_exc()
    
    # Sort by position
    markers.sort(key=lambda x: x['milliseconds'])
    
    if markers:
        logging.info(f"Extracted {len(markers)} markers from {os.path.basename(wav_file)}")
        logging.info(f"First marker: {markers[0]['timestamp']} ({markers[0]['breathphase']})")
        logging.info(f"Last marker: {markers[-1]['timestamp']} ({markers[-1]['breathphase']})")
    else:
        logging.warning(f"No markers found in {os.path.basename(wav_file)}")
    
    return markers

def concatenate_audio_files(output_path, *files):
    """Concatenate audio files and combine their markers."""
    # Check for valid files
    valid_files = [f for f in files if os.path.exists(f)]
    if not valid_files:
        raise ValueError("No valid audio files to concatenate")
    
    # Read audio segments
    segments = []
    for file in valid_files:
        segments.append(AudioSegment.from_wav(file))
    
    # Concatenate segments
    combined = segments[0]
    for segment in segments[1:]:
        combined += segment
    
    # Create directory if needed
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Save concatenated audio
    combined.export(output_path, format="wav")
    
    logging.info(f"Created concatenated audio file: {output_path}")
    
    # Extract markers from the concatenated file
    markers = extract_markers_from_wav(output_path)
    
    return output_path, markers

def create_markers_csv(markers, csv_path):
    """Create a CSV file with marker information."""
    # Convert to DataFrame
    df = pd.DataFrame(markers)
    
    # Add description column
    df['description'] = df.apply(
        lambda row: f"{row['breathphase'].capitalize()} hold marker", 
        axis=1
    )
    
    # Rename columns for consistency with expected format
    df = df.rename(columns={
        'timestamp': 'retentionslot_timestamp',
    })
    
    # Save CSV
    df.to_csv(csv_path, index=False)
    
    logging.info(f"Created marker CSV: {csv_path}")
    
    return csv_path, df

# ----------------------------------------------------------------------
# Main Generator Class
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates PPS experiment designs based on breath markers in WAV files."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare directories
        self.exp_log_dir = self.config['paths']['experiment_log_dir']
        self.audio_dir = self.config['paths']['audio_output_dir']
        os.makedirs(self.exp_log_dir, exist_ok=True)
        os.makedirs(self.audio_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Initialize randomization
        self.base_seed = 42
        random.seed(self.base_seed)
        np.random.seed(self.base_seed)
        
        # Create or find audio and extract markers
        self._prepare_audio_and_markers()
    
    def _calculate_trial_counts(self):
        """Calculate number of trials for each condition."""
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        logging.info("Trial count calculation:")
        for cond, count in trial_counts.items():
            logging.info(f"  {cond}: {count} trials")
        logging.info(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _prepare_audio_and_markers(self):
        """Create concatenated audio file and extract markers."""
        # Calculate total required markers
        total_trials = sum(self.trial_counts.values())
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = total_trials + practice_trials
        
        # Look for existing concatenated files
        existing_files = []
        if os.path.exists(self.audio_dir):
            existing_files = [f for f in os.listdir(self.audio_dir) 
                             if f.startswith("BoxBreathing_complete_") and f.endswith(".wav")]
        
        # Try to use existing file if it has enough markers
        self.concat_path = None
        self.markers = None
        self.markers_csv_path = None
        
        if existing_files:
            for file in sorted(existing_files, reverse=True):  # Try newest first
                try:
                    file_path = os.path.join(self.audio_dir, file)
                    markers = extract_markers_from_wav(file_path)
                    
                    if len(markers) >= required_markers:
                        self.concat_path = file_path
                        self.markers = markers
                        
                        # Check for CSV
                        csv_path = file_path.replace('.wav', '_markers.csv')
                        if os.path.exists(csv_path):
                            self.markers_csv_path = csv_path
                            logging.info(f"Using existing marker CSV: {csv_path}")
                        else:
                            # Create CSV if not exists
                            self.markers_csv_path, _ = create_markers_csv(markers, csv_path)
                        
                        logging.info(f"Using existing audio file: {file}")
                        logging.info(f"  - Contains {len(markers)} markers, need {required_markers}")
                        break
                except Exception as e:
                    logging.warning(f"Error reading markers from {file}: {e}")
        
        # Create new file if needed
        if self.concat_path is None:
            logging.info("Creating new concatenated audio file...")
            self._create_concatenated_audio(required_markers)
        
        # Load marker data for trial assignment
        if self.markers_csv_path:
            self.markers_df = pd.read_csv(self.markers_csv_path)
        else:
            # Fallback - create DataFrame from markers
            self.markers_df = pd.DataFrame(self.markers)
            self.markers_df['description'] = self.markers_df.apply(
                lambda row: f"{row['breathphase'].capitalize()} hold marker", 
                axis=1
            )
    
    def _create_concatenated_audio(self, required_markers):
        """Create concatenated audio with sufficient markers."""
        intro_file = self.config['paths']['breathing_intro']
        segment_file = self.config['paths']['breathing_segment']
        outro_file = self.config['paths']['breathing_outro']
        
        # Check source files
        if not os.path.exists(intro_file):
            raise FileNotFoundError(f"Intro file not found: {intro_file}")
        if not os.path.exists(segment_file):
            raise FileNotFoundError(f"Segment file not found: {segment_file}")
        
        # Get marker counts from source files
        intro_markers = extract_markers_from_wav(intro_file)
        segment_markers = extract_markers_from_wav(segment_file)
        
        if not intro_markers:
            raise ValueError(f"No markers found in intro file: {intro_file}")
        if not segment_markers:
            raise ValueError(f"No markers found in segment file: {segment_file}")
        
        # Calculate needed segments
        intro_count = len(intro_markers)
        segment_count = len(segment_markers)
        remaining = max(0, required_markers - intro_count)
        segments_needed = math.ceil(remaining / segment_count)
        
        logging.info(f"Creating concatenated audio for {required_markers} markers:")
        logging.info(f"  Intro markers: {intro_count}")
        logging.info(f"  Segment markers: {segment_count}")
        logging.info(f"  Segments needed: {segments_needed}")
        
        # Build file list
        files_to_concat = [intro_file]
        for _ in range(segments_needed):
            files_to_concat.append(segment_file)
        
        if os.path.exists(outro_file):
            files_to_concat.append(outro_file)
        
        # Create output path
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        self.concat_path = os.path.join(
            self.audio_dir,
            f"BoxBreathing_complete_x{segments_needed}_{timestamp}.wav"
        )
        
        # Concatenate files
        self.concat_path, self.markers = concatenate_audio_files(
            self.concat_path,
            *files_to_concat
        )
        
        # Create marker CSV
        self.markers_csv_path, self.markers_df = create_markers_csv(
            self.markers,
            self.concat_path.replace('.wav', '_markers.csv')
        )
    
    def generate_counterbalanced_design(self, participant_id):
        """Generate counterbalanced trial conditions."""
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = list(LOOMING_STIMULI.keys())
        repetitions = self.config['repetitions']
        
        all_trials = []
        
        # Generate main trials for each condition
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate trials
            for soa in soa_conditions:
                # Distribute among stimulus types
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle
                random.shuffle(stim_trials)
                
                # Create trials
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True
                    })
        
        # Add catch trials
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly
            stim_counts = {stim: 0 for stim in stimulus_types}
            catch_trials = []
            
            for _ in range(catch_count):
                min_stim = min(stim_counts, key=stim_counts.get)
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                stim_counts[min_stim] += 1
            
            # Add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        return all_trials_df
    
    def assign_trials_to_markers(self, design_df, participant_id):
        """Assign trials to markers based on breathing phase requirements."""
        # Account for practice trials
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = len(design_df) + practice_trials
        
        # Verify we have enough markers
        if len(self.markers) < required_markers:
            raise ValueError(f"Not enough markers ({len(self.markers)}) for required trials ({required_markers})")
        
        # Get copy of markers DataFrame
        markers_df = self.markers_df.copy()
        
        # Sort by position/milliseconds
        if 'milliseconds' in markers_df.columns:
            markers_df = markers_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            markers_df = markers_df.sort_values(by='position').reset_index(drop=True)
        
        # Reserve practice slots
        practice_markers = markers_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_markers = markers_df.iloc[practice_trials:practice_trials+len(design_df)].copy()
        
        # Create practice trials
        practice_trial_list = []
        if practice_trials > 0:
            for i, row in practice_markers.iterrows():
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),
                    'is_tactile': False
                }
                
                # Add all marker columns
                for col in row.index:
                    practice_trial[col] = row[col]
                
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count marker types
        inhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'inhale']
        exhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'exhale']
        n_inhale = len(inhale_markers)
        n_exhale = len(exhale_markers)
        
        logging.info(f"Markers: {n_inhale} inhale, {n_exhale} exhale")
        logging.info(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, "
                    f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough of each type
        if len(pool_inh) > n_inhale:
            raise ValueError(f"Not enough inhale markers ({n_inhale}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale:
            raise ValueError(f"Not enough exhale markers ({n_exhale}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining markers
        remaining_inhale = n_inhale - len(pool_inh)
        remaining_exhale = n_exhale - len(pool_exh)
        total_flex = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough total markers
        if remaining_inhale + remaining_exhale < total_flex:
            raise ValueError(f"Not enough remaining markers ({remaining_inhale + remaining_exhale})"
                            f" for flexible trials ({total_flex})")
        
        # Distribute baseline and catch trials
        baseline_inhale = min(len(pool_baseline) // 2, remaining_inhale)
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = min(len(pool_catch) // 2, remaining_inhale - baseline_inhale)
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create phase pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        logging.info(f"Distributing flexible trials:")
        logging.info(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
        logging.info(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Final pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Map phase to pools
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Assign trials to markers
        regular_assignments = []
        
        for _, row in regular_markers.iterrows():
            phase = row['breathphase'].lower()
            trial_pool = phase_to_pool[phase]
            
            if trial_pool:  # If we still have trials of this phase
                trial = trial_pool.pop(0)  # Get next trial
                
                # Combine trial with marker
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                regular_assignments.append(combined)
        
        # Combine practice and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final DataFrame
        final_df = pd.DataFrame(final_assignments)
        
        # Sort by marker position
        if 'milliseconds' in final_df.columns:
            final_df = final_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            final_df = final_df.sort_values(by='position').reset_index(drop=True)
        
        # Renumber trials
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Add jittered timestamps
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df
    
    def finalize_design_csv(self, design_df):
        """Finalize the design CSV with proper formatting."""
        # Set appropriate null values based on trial type
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected_response flag
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        # Determine column order
        marker_cols = ['retentionslot_timestamp' if 'retentionslot_timestamp' in design_df.columns else 'timestamp', 
                       'milliseconds', 'sample_index' if 'sample_index' in design_df.columns else 'position', 
                       'description', 'breathphase']
        trial_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 
                      'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 
                      'soa_ms', 'jittered_ms', 'expected_response']
        
        # Combine and reorder columns
        all_cols = []
        for col in marker_cols:
            if col in design_df.columns:
                all_cols.append(col)
        for col in trial_cols:
            if col in design_df.columns and col not in all_cols:
                all_cols.append(col)
        remaining_cols = [col for col in design_df.columns if col not in all_cols]
        final_order = all_cols + remaining_cols
        
        # Apply column order
        design_df = design_df[final_order]
        
        return design_df
    
    def process_participant(self, participant_id):
        """Process a single participant."""
        try:
            logging.info(f"=== Processing participant {participant_id} ===")
            
            # Generate counterbalanced design
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # Assign to markers
            design_df = self.assign_trials_to_markers(design_df, participant_id)
            
            # Finalize design
            design_df = self.finalize_design_csv(design_df)
            
            # Save to CSV
            design_path = os.path.join(self.exp_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(design_path, index=False)
            
            logging.info(f"Saved design CSV: {design_path}")
            
            return True, {'design_path': design_path}
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """Generate designs for multiple participants."""
        if participant_ids is None:
            participant_ids = list(range(self.config['num_participants']))
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        return results

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def main():
    """Main function."""
    # Print configuration
    print("\n===== PPS DESIGN GENERATOR FOR PRE-MARKED WAV FILES =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {list(LOOMING_STIMULI.keys())}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    total_trials = total_base_trials + catch_trials + practice_trials
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_trials}")
    
    # Get number of participants from config
    num_participants = CONFIG['num_participants']
    print(f"\nGenerating designs for {num_participants} participants...")
    
    # Create generator and run
    try:
        generator = PPSDesignGenerator(CONFIG)
        results = generator.run()
        
        # Summary
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"\n===== GENERATION SUMMARY =====")
        print(f"Successfully generated {success_count} of {len(results)} participant designs")
        print(f"Audio file: {os.path.basename(generator.concat_path)}")
        print(f"Marker CSV: {os.path.basename(generator.markers_csv_path)}")
        
        print("\nParticipant CSV files:")
        for pid, info in results.items():
            if info['success']:
                print(f"  Participant {pid}: {os.path.basename(info['paths']['design_path'])}")
            else:
                print(f"  Participant {pid}: ERROR - failed to generate")
        
    except Exception as e:
        print(f"Error during design generation: {e}")
        traceback.print_exc()
    
    print("\nDesign generation complete.")
    print("The concatenated audio file and participant designs have been saved.")
    print("Use these files with the audio generator to create the final stimulus files.")

if __name__ == "__main__":
    main()

2025-03-27 20:08:34,831 - INFO - Trial count calculation:
2025-03-27 20:08:34,832 - INFO -   inhalation: 60 trials
2025-03-27 20:08:34,836 - INFO -   exhalation: 60 trials
2025-03-27 20:08:34,837 - INFO -   baseline: 60 trials
2025-03-27 20:08:34,840 - INFO -   catch: 18 trials
2025-03-27 20:08:34,842 - INFO -   Total: 198 trials
2025-03-27 20:08:34,845 - INFO - Creating new concatenated audio file...
2025-03-27 20:08:34,848 - INFO - Found 38 cue points in BoxBreathing_intro_with_markers.wav
2025-03-27 20:08:34,849 - INFO - Extracted 38 markers from BoxBreathing_intro_with_markers.wav
2025-03-27 20:08:34,851 - INFO - First marker: 00:31.9 (inhale)
2025-03-27 20:08:34,853 - INFO - Last marker: 05:28.5 (exhale)
2025-03-27 20:08:34,855 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:08:34,856 - INFO - Extracted 34 markers from BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:08:34,857 - INFO - First marker: 00:7.1 (inhale)
2025-03


===== PPS DESIGN GENERATOR FOR PRE-MARKED WAV FILES =====
Repetitions (trials per SOA per condition): 12
Practice trials: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206

Generating designs for 100 participants...


2025-03-27 20:08:36,133 - INFO - Created concatenated audio file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete_x5_20250327_200834.wav
2025-03-27 20:08:36,150 - WARNING - No markers found in BoxBreathing_complete_x5_20250327_200834.wav


Error during design generation: Cannot set a DataFrame without columns to the column description

Design generation complete.
The concatenated audio file and participant designs have been saved.
Use these files with the audio generator to create the final stimulus files.


Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3436667614.py", line 740, in main
    generator = PPSDesignGenerator(CONFIG)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3436667614.py", line 253, in __init__
    self._prepare_audio_and_markers()
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3436667614.py", line 324, in _prepare_audio_and_markers
    self._create_concatenated_audio(required_markers)
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3436667614.py", line 391, in _create_concatenated_audio
    self.markers_csv_path, self.markers_df = create_markers_csv(
                                             ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\3436667614.py", line 211, in create_markers_csv
    df['description'] = df.apply(
    ~~^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\

In [5]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Simplified PPS Design Generator for Pre-Marked WAV Files

This script generates participant-specific experimental design CSVs for a Peripersonal 
Space (PPS) experiment using WAV files that already have markers embedded.

The workflow:
1. Concatenate existing WAV files that already have breath markers embedded
2. Extract all markers from the concatenated WAV file into a template CSV
3. Generate participant-specific design CSVs by:
   a. Creating counterbalanced trial conditions
   b. Assigning conditions to markers based on breathphase constraints
   c. Computing stimulus timing information
4. Save the concatenated WAV file and all CSVs in their designated locations
"""

import os
import wave
import struct
import numpy as np
import pandas as pd
import random
import traceback
import json
import re
import math
from pathlib import Path
from pydub import AudioSegment
import io
import logging

# Set up logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory for output files
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'audio_output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        # Input audio files with breath markers
        'breathing_intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav",
        'breathing_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment_with_markers.wav",
        'breathing_outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
    },
    'box_breathing': {
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# Marker type constants
MARKER_TYPE_INHALATION = 1  # Odd IDs = Inhale
MARKER_TYPE_EXHALATION = 2  # Even IDs = Exhale

# ----------------------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    """Convert milliseconds to timestamp in format MM:SS.S."""
    minutes = int(ms // 60000)
    seconds = (ms % 60000) / 1000.0
    return f"{minutes:02d}:{seconds:.1f}"

def extract_markers_from_wav(wav_file):
    """Extract marker information directly from a WAV file's cue points."""
    markers = []
    
    try:
        # First get sample rate
        with wave.open(wav_file, 'rb') as wav:
            sample_rate = wav.getframerate()
        
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Skip file size
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find cue chunk
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_header == b'cue ':
                    # Found cue chunk
                    num_cue_points = int.from_bytes(f.read(4), byteorder='little')
                    logging.info(f"Found {num_cue_points} cue points in {os.path.basename(wav_file)}")
                    
                    # Read all cue points
                    for i in range(num_cue_points):
                        # Read cue ID
                        cue_id = int.from_bytes(f.read(4), byteorder='little')
                        # Read position
                        position = int.from_bytes(f.read(4), byteorder='little')
                        # Skip remaining cue point data (16 bytes)
                        f.read(16)
                        
                        # Calculate timestamp
                        position_ms = (position * 1000) / sample_rate
                        
                        # Determine breathphase from cue ID parity
                        breathphase = "inhale" if cue_id % 2 == 1 else "exhale"
                        
                        markers.append({
                            'id': cue_id,
                            'position': position,
                            'milliseconds': position_ms,
                            'timestamp': convert_ms_to_timestamp(position_ms),
                            'breathphase': breathphase,
                            'sample_rate': sample_rate
                        })
                    
                    break  # Exit loop after finding cue chunk
                else:
                    # Skip to next chunk
                    f.seek(chunk_pos + 8 + chunk_size)
    
    except Exception as e:
        logging.error(f"Error reading markers from {os.path.basename(wav_file)}: {e}")
        traceback.print_exc()
    
    # Sort by position
    markers.sort(key=lambda x: x['milliseconds'])
    
    if markers:
        logging.info(f"Extracted {len(markers)} markers from {os.path.basename(wav_file)}")
        logging.info(f"First marker: {markers[0]['timestamp']} ({markers[0]['breathphase']})")
        logging.info(f"Last marker: {markers[-1]['timestamp']} ({markers[-1]['breathphase']})")
    else:
        logging.warning(f"No markers found in {os.path.basename(wav_file)}")
    
    return markers

def concatenate_audio_files(output_path, *files):
    """Concatenate audio files and combine their markers."""
    # Check for valid files
    valid_files = [f for f in files if os.path.exists(f)]
    if not valid_files:
        raise ValueError("No valid audio files to concatenate")
    
    # Read audio segments
    segments = []
    for file in valid_files:
        segments.append(AudioSegment.from_wav(file))
    
    # Concatenate segments
    combined = segments[0]
    for segment in segments[1:]:
        combined += segment
    
    # Create directory if needed
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Save concatenated audio
    combined.export(output_path, format="wav")
    
    logging.info(f"Created concatenated audio file: {output_path}")
    
    # Extract markers from the concatenated file
    markers = extract_markers_from_wav(output_path)
    
    return output_path, markers

def create_markers_csv(markers, csv_path):
    """Create a CSV file with marker information."""
    # Convert to DataFrame
    df = pd.DataFrame(markers)
    
    # Add description column if not present
    if 'description' not in df.columns:
        df['description'] = df.apply(
            lambda row: f"{row['breathphase'].capitalize()} hold marker", 
            axis=1
        )
    
    # Rename columns for consistency with expected format
    if 'timestamp' in df.columns and 'retentionslot_timestamp' not in df.columns:
        df = df.rename(columns={
            'timestamp': 'retentionslot_timestamp',
        })
    
    # Make sure we have all required columns
    required_columns = ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
    for col in required_columns:
        if col not in df.columns:
            if col == 'sample_index' and 'position' in df.columns:
                df['sample_index'] = df['position']
            elif col == 'retentionslot_timestamp' and 'timestamp' in df.columns:
                df['retentionslot_timestamp'] = df['timestamp']
            else:
                logging.warning(f"Missing required column: {col}")
    
    # Save CSV
    df.to_csv(csv_path, index=False)
    
    logging.info(f"Created marker CSV: {csv_path}")
    
    return csv_path, df

# ----------------------------------------------------------------------
# Main Generator Class
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates PPS experiment designs based on breath markers in WAV files."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare directories
        self.exp_log_dir = self.config['paths']['experiment_log_dir']
        self.audio_dir = self.config['paths']['audio_output_dir']
        os.makedirs(self.exp_log_dir, exist_ok=True)
        os.makedirs(self.audio_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Initialize randomization
        self.base_seed = 42
        random.seed(self.base_seed)
        np.random.seed(self.base_seed)
        
        # Create or find audio and extract markers
        self._prepare_audio_and_markers()
    
    def _calculate_trial_counts(self):
        """Calculate number of trials for each condition."""
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        logging.info("Trial count calculation:")
        for cond, count in trial_counts.items():
            logging.info(f"  {cond}: {count} trials")
        logging.info(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _prepare_audio_and_markers(self):
        """Create concatenated audio file and extract markers."""
        # Calculate total required markers
        total_trials = sum(self.trial_counts.values())
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = total_trials + practice_trials
        
        # Look for existing concatenated files
        existing_files = []
        if os.path.exists(self.audio_dir):
            existing_files = [f for f in os.listdir(self.audio_dir) 
                             if f.startswith("BoxBreathing_complete_") and f.endswith(".wav")]
        
        # Try to use existing file if it has enough markers
        self.concat_path = None
        self.markers = None
        self.markers_csv_path = None
        
        if existing_files:
            for file in sorted(existing_files, reverse=True):  # Try newest first
                try:
                    file_path = os.path.join(self.audio_dir, file)
                    markers = extract_markers_from_wav(file_path)
                    
                    if len(markers) >= required_markers:
                        self.concat_path = file_path
                        self.markers = markers
                        
                        # Check for existing CSV with same base name
                        csv_base_name = os.path.splitext(file)[0]
                        possible_csv_paths = [
                            os.path.join(self.audio_dir, f"{csv_base_name}_markers.csv"),
                            # Check for alternative naming patterns
                            os.path.join(self.audio_dir, f"{csv_base_name}.csv")
                        ]
                        
                        # Find existing CSV or create one
                        csv_found = False
                        for csv_path in possible_csv_paths:
                            if os.path.exists(csv_path):
                                self.markers_csv_path = csv_path
                                logging.info(f"Using existing marker CSV: {os.path.basename(csv_path)}")
                                csv_found = True
                                break
                        
                        if not csv_found:
                            # Create new CSV if none found
                            self.markers_csv_path, _ = create_markers_csv(
                                markers, 
                                os.path.join(self.audio_dir, f"{csv_base_name}_markers.csv")
                            )
                        
                        logging.info(f"Using existing audio file: {file}")
                        logging.info(f"  - Contains {len(markers)} markers, need {required_markers}")
                        break
                except Exception as e:
                    logging.warning(f"Error reading markers from {file}: {e}")
        
        # Create new file if needed
        if self.concat_path is None:
            logging.info("Creating new concatenated audio file...")
            self._create_concatenated_audio(required_markers)
        
        # Load marker data for trial assignment
        if self.markers_csv_path and os.path.exists(self.markers_csv_path):
            # Load CSV file
            self.markers_df = pd.read_csv(self.markers_csv_path)
            logging.info(f"Loaded {len(self.markers_df)} markers from CSV: {os.path.basename(self.markers_csv_path)}")
        else:
            # Fallback - create DataFrame from markers
            self.markers_df = pd.DataFrame(self.markers)
            self.markers_df['description'] = self.markers_df.apply(
                lambda row: f"{row['breathphase'].capitalize()} hold marker", 
                axis=1
            )
            logging.warning("Created markers DataFrame from WAV markers (no CSV found).")
    
    def _create_concatenated_audio(self, required_markers):
        """Create concatenated audio with sufficient markers."""
        intro_file = self.config['paths']['breathing_intro']
        segment_file = self.config['paths']['breathing_segment']
        outro_file = self.config['paths']['breathing_outro']
        
        # Check source files
        if not os.path.exists(intro_file):
            raise FileNotFoundError(f"Intro file not found: {intro_file}")
        if not os.path.exists(segment_file):
            raise FileNotFoundError(f"Segment file not found: {segment_file}")
        
        # Get marker counts from source files
        intro_markers = extract_markers_from_wav(intro_file)
        segment_markers = extract_markers_from_wav(segment_file)
        
        if not intro_markers:
            raise ValueError(f"No markers found in intro file: {intro_file}")
        if not segment_markers:
            raise ValueError(f"No markers found in segment file: {segment_file}")
        
        # Calculate needed segments
        intro_count = len(intro_markers)
        segment_count = len(segment_markers)
        remaining = max(0, required_markers - intro_count)
        segments_needed = math.ceil(remaining / segment_count)
        
        logging.info(f"Creating concatenated audio for {required_markers} markers:")
        logging.info(f"  Intro markers: {intro_count}")
        logging.info(f"  Segment markers: {segment_count}")
        logging.info(f"  Segments needed: {segments_needed}")
        
        # Build file list
        files_to_concat = [intro_file]
        for _ in range(segments_needed):
            files_to_concat.append(segment_file)
        
        if os.path.exists(outro_file):
            files_to_concat.append(outro_file)
        
        # Create output path
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        self.concat_path = os.path.join(
            self.audio_dir,
            f"BoxBreathing_complete_x{segments_needed}_{timestamp}.wav"
        )
        
        # Concatenate files
        self.concat_path, self.markers = concatenate_audio_files(
            self.concat_path,
            *files_to_concat
        )
        
        # Create marker CSV
        self.markers_csv_path, self.markers_df = create_markers_csv(
            self.markers,
            self.concat_path.replace('.wav', '_markers.csv')
        )
    
    def generate_counterbalanced_design(self, participant_id):
        """Generate counterbalanced trial conditions."""
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = list(LOOMING_STIMULI.keys())
        repetitions = self.config['repetitions']
        
        all_trials = []
        
        # Generate main trials for each condition
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate trials
            for soa in soa_conditions:
                # Distribute among stimulus types
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle
                random.shuffle(stim_trials)
                
                # Create trials
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True
                    })
        
        # Add catch trials
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly
            stim_counts = {stim: 0 for stim in stimulus_types}
            catch_trials = []
            
            for _ in range(catch_count):
                min_stim = min(stim_counts, key=stim_counts.get)
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                stim_counts[min_stim] += 1
            
            # Add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        return all_trials_df
    
    def assign_trials_to_markers(self, design_df, participant_id):
        """Assign trials to markers based on breathing phase requirements."""
        # Account for practice trials
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = len(design_df) + practice_trials
        
        # Verify we have enough markers
        if len(self.markers) < required_markers:
            raise ValueError(f"Not enough markers ({len(self.markers)}) for required trials ({required_markers})")
        
        # Get copy of markers DataFrame
        markers_df = self.markers_df.copy()
        
        # Sort by position/milliseconds
        if 'milliseconds' in markers_df.columns:
            markers_df = markers_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            markers_df = markers_df.sort_values(by='position').reset_index(drop=True)
        
        # Reserve practice slots
        practice_markers = markers_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_markers = markers_df.iloc[practice_trials:practice_trials+len(design_df)].copy()
        
        # Create practice trials
        practice_trial_list = []
        if practice_trials > 0:
            for i, row in practice_markers.iterrows():
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),
                    'is_tactile': False
                }
                
                # Add all marker columns
                for col in row.index:
                    practice_trial[col] = row[col]
                
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count marker types
        inhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'inhale']
        exhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'exhale']
        n_inhale = len(inhale_markers)
        n_exhale = len(exhale_markers)
        
        logging.info(f"Markers: {n_inhale} inhale, {n_exhale} exhale")
        logging.info(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, "
                    f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough of each type
        if len(pool_inh) > n_inhale:
            raise ValueError(f"Not enough inhale markers ({n_inhale}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale:
            raise ValueError(f"Not enough exhale markers ({n_exhale}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining markers
        remaining_inhale = n_inhale - len(pool_inh)
        remaining_exhale = n_exhale - len(pool_exh)
        total_flex = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough total markers
        if remaining_inhale + remaining_exhale < total_flex:
            raise ValueError(f"Not enough remaining markers ({remaining_inhale + remaining_exhale})"
                            f" for flexible trials ({total_flex})")
        
        # Distribute baseline and catch trials
        baseline_inhale = min(len(pool_baseline) // 2, remaining_inhale)
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = min(len(pool_catch) // 2, remaining_inhale - baseline_inhale)
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create phase pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        logging.info(f"Distributing flexible trials:")
        logging.info(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
        logging.info(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Final pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Map phase to pools
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Assign trials to markers
        regular_assignments = []
        
        for _, row in regular_markers.iterrows():
            phase = row['breathphase'].lower()
            trial_pool = phase_to_pool[phase]
            
            if trial_pool:  # If we still have trials of this phase
                trial = trial_pool.pop(0)  # Get next trial
                
                # Combine trial with marker
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                regular_assignments.append(combined)
        
        # Combine practice and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final DataFrame
        final_df = pd.DataFrame(final_assignments)
        
        # Sort by marker position
        if 'milliseconds' in final_df.columns:
            final_df = final_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            final_df = final_df.sort_values(by='position').reset_index(drop=True)
        
        # Renumber trials
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Add jittered timestamps
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(convert_ms_to_timestamp)
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(convert_ms_to_timestamp)
        
        return final_df
    
    def finalize_design_csv(self, design_df):
        """Finalize the design CSV with proper formatting."""
        # Set appropriate null values based on trial type
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected_response flag
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        # Determine column order
        marker_cols = ['retentionslot_timestamp' if 'retentionslot_timestamp' in design_df.columns else 'timestamp', 
                       'milliseconds', 'sample_index' if 'sample_index' in design_df.columns else 'position', 
                       'description', 'breathphase']
        trial_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 
                      'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 
                      'soa_ms', 'jittered_ms', 'expected_response']
        
        # Combine and reorder columns
        all_cols = []
        for col in marker_cols:
            if col in design_df.columns:
                all_cols.append(col)
        for col in trial_cols:
            if col in design_df.columns and col not in all_cols:
                all_cols.append(col)
        remaining_cols = [col for col in design_df.columns if col not in all_cols]
        final_order = all_cols + remaining_cols
        
        # Apply column order
        design_df = design_df[final_order]
        
        return design_df
    
    def process_participant(self, participant_id):
        """Process a single participant."""
        try:
            logging.info(f"=== Processing participant {participant_id} ===")
            
            # Generate counterbalanced design
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # Assign to markers
            design_df = self.assign_trials_to_markers(design_df, participant_id)
            
            # Finalize design
            design_df = self.finalize_design_csv(design_df)
            
            # Save to CSV
            design_path = os.path.join(self.exp_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(design_path, index=False)
            
            logging.info(f"Saved design CSV: {design_path}")
            
            return True, {'design_path': design_path}
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """Generate designs for multiple participants."""
        if participant_ids is None:
            participant_ids = list(range(self.config['num_participants']))
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        return results

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def main():
    """Main function."""
    # Print configuration
    print("\n===== PPS DESIGN GENERATOR FOR PRE-MARKED WAV FILES =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {list(LOOMING_STIMULI.keys())}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    total_trials = total_base_trials + catch_trials + practice_trials
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_trials}")
    
    # Get number of participants from config
    num_participants = CONFIG['num_participants']
    print(f"\nGenerating designs for {num_participants} participants...")
    
    # Create generator and run
    try:
        generator = PPSDesignGenerator(CONFIG)
        results = generator.run()
        
        # Summary
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"\n===== GENERATION SUMMARY =====")
        print(f"Successfully generated {success_count} of {len(results)} participant designs")
        print(f"Audio file: {os.path.basename(generator.concat_path)}")
        print(f"Marker CSV: {os.path.basename(generator.markers_csv_path)}")
        
        print("\nParticipant CSV files:")
        for pid, info in results.items():
            if info['success']:
                print(f"  Participant {pid}: {os.path.basename(info['paths']['design_path'])}")
            else:
                print(f"  Participant {pid}: ERROR - failed to generate")
        
    except Exception as e:
        print(f"Error during design generation: {e}")
        traceback.print_exc()
    
    print("\nDesign generation complete.")
    print("The concatenated audio file and participant designs have been saved.")
    print("Use these files with the audio generator to create the final stimulus files.")

if __name__ == "__main__":
    main()

2025-03-27 20:15:56,396 - INFO - Trial count calculation:
2025-03-27 20:15:56,396 - INFO -   inhalation: 60 trials
2025-03-27 20:15:56,397 - INFO -   exhalation: 60 trials
2025-03-27 20:15:56,398 - INFO -   baseline: 60 trials
2025-03-27 20:15:56,399 - INFO -   catch: 18 trials
2025-03-27 20:15:56,399 - INFO -   Total: 198 trials
2025-03-27 20:15:56,401 - WARNING - No markers found in BoxBreathing_complete_x5_20250327_200834.wav
2025-03-27 20:15:56,401 - INFO - Creating new concatenated audio file...
2025-03-27 20:15:56,404 - INFO - Found 38 cue points in BoxBreathing_intro_with_markers.wav
2025-03-27 20:15:56,404 - INFO - Extracted 38 markers from BoxBreathing_intro_with_markers.wav
2025-03-27 20:15:56,404 - INFO - First marker: 00:31.9 (inhale)
2025-03-27 20:15:56,405 - INFO - Last marker: 05:28.5 (exhale)
2025-03-27 20:15:56,406 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:15:56,408 - INFO - Extracted 34 markers from BoxBreathing_repe


===== PPS DESIGN GENERATOR FOR PRE-MARKED WAV FILES =====
Repetitions (trials per SOA per condition): 12
Practice trials: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206

Generating designs for 100 participants...


2025-03-27 20:15:57,698 - INFO - Created concatenated audio file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete_x5_20250327_201556.wav
2025-03-27 20:15:57,711 - WARNING - No markers found in BoxBreathing_complete_x5_20250327_201556.wav


Error during design generation: Cannot set a DataFrame without columns to the column description

Design generation complete.
The concatenated audio file and participant designs have been saved.
Use these files with the audio generator to create the final stimulus files.


Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\497693486.py", line 771, in main
    generator = PPSDesignGenerator(CONFIG)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\497693486.py", line 266, in __init__
    self._prepare_audio_and_markers()
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\497693486.py", line 352, in _prepare_audio_and_markers
    self._create_concatenated_audio(required_markers)
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\497693486.py", line 422, in _create_concatenated_audio
    self.markers_csv_path, self.markers_df = create_markers_csv(
                                             ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_23572\497693486.py", line 212, in create_markers_csv
    df['description'] = df.apply(
    ~~^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-

In [6]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Direct WAV Concatenation with Marker CSV Generation

This script:
1. Creates a concatenated WAV file from source files with markers
2. Directly creates a CSV with calculated marker positions
3. Generates participant-specific design CSVs

The key issue with the previous approach was that markers weren't being preserved
during concatenation. This version manually tracks and calculates marker positions.
"""

import os
import wave
import struct
import numpy as np
import pandas as pd
import random
import traceback
import datetime
from pydub import AudioSegment
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory for output files
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'audio_output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        # Input audio files with breath markers
        'breathing_intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav",
        'breathing_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment_with_markers.wav",
        'breathing_outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
    },
    'box_breathing': {
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# ----------------------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    """Convert milliseconds to timestamp in format MM:SS.S."""
    minutes = int(ms // 60000)
    seconds = (ms % 60000) / 1000.0
    return f"{minutes:02d}:{seconds:.1f}"

def extract_markers_from_wav(wav_file):
    """Extract marker information from a WAV file's cue points."""
    markers = []
    
    try:
        # Get sample rate
        with wave.open(wav_file, 'rb') as wav:
            sample_rate = wav.getframerate()
            frames = wav.getnframes()
            duration_ms = (frames / sample_rate) * 1000
        
        # Open file for binary reading
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Skip file size
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find cue chunk
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_header == b'cue ':
                    # Found cue chunk
                    num_cues = int.from_bytes(f.read(4), byteorder='little')
                    logging.info(f"Found {num_cues} cue points in {os.path.basename(wav_file)}")
                    
                    # Read all cue points
                    for i in range(num_cues):
                        # Read cue ID and position
                        cue_id = int.from_bytes(f.read(4), byteorder='little')
                        position = int.from_bytes(f.read(4), byteorder='little')
                        
                        # Skip remaining cue point data
                        f.read(16)
                        
                        # Calculate timestamp
                        position_ms = (position / sample_rate) * 1000
                        
                        # Determine breathphase from ID parity
                        breathphase = "inhale" if cue_id % 2 == 1 else "exhale"
                        
                        markers.append({
                            'id': cue_id,
                            'position': position,
                            'milliseconds': position_ms,
                            'retentionslot_timestamp': convert_ms_to_timestamp(position_ms),
                            'breathphase': breathphase,
                            'description': f"{breathphase.capitalize()} hold marker"
                        })
                    
                    break
                else:
                    # Skip to next chunk
                    f.seek(chunk_pos + 8 + chunk_size)
        
        # Sort markers by position
        markers.sort(key=lambda x: x['position'])
        
        if markers:
            logging.info(f"Extracted {len(markers)} markers from {os.path.basename(wav_file)}")
            logging.info(f"First marker: {markers[0]['retentionslot_timestamp']} ({markers[0]['breathphase']})")
            logging.info(f"Last marker: {markers[-1]['retentionslot_timestamp']} ({markers[-1]['breathphase']})")
        else:
            logging.warning(f"No markers found in {os.path.basename(wav_file)}")
        
    except Exception as e:
        logging.error(f"Error extracting markers from {os.path.basename(wav_file)}: {e}")
        traceback.print_exc()
    
    return markers

def concatenate_audio_with_markers(output_wav_path, output_csv_path, files_to_concat):
    """
    Concatenate audio files and manually create a marker CSV.
    
    Args:
        output_wav_path: Path to save the concatenated WAV file
        output_csv_path: Path to save the marker CSV file
        files_to_concat: List of WAV files to concatenate
    
    Returns:
        Tuple of (WAV path, marker list, CSV path)
    """
    # Check input files
    valid_files = [f for f in files_to_concat if os.path.exists(f)]
    if not valid_files:
        raise ValueError("No valid audio files to concatenate")
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_wav_path), exist_ok=True)
    
    # Extract markers from each file
    all_markers = []
    audio_segments = []
    current_offset_ms = 0
    
    for file_path in valid_files:
        # Load audio segment
        audio = AudioSegment.from_wav(file_path)
        audio_segments.append(audio)
        
        # Get markers and adjust positions
        markers = extract_markers_from_wav(file_path)
        
        # Adjust marker positions based on offset
        for marker in markers:
            marker['milliseconds'] += current_offset_ms
            marker['retentionslot_timestamp'] = convert_ms_to_timestamp(marker['milliseconds'])
            marker['position'] += int(current_offset_ms * audio.frame_rate / 1000)
            marker['sample_index'] = marker['position']  # Add sample_index for compatibility
        
        # Add adjusted markers
        all_markers.extend(markers)
        
        # Update offset for next file
        current_offset_ms += len(audio)
    
    # Concatenate audio
    concatenated_audio = audio_segments[0]
    for segment in audio_segments[1:]:
        concatenated_audio += segment
    
    # Save concatenated WAV
    concatenated_audio.export(output_wav_path, format="wav")
    logging.info(f"Created concatenated audio: {output_wav_path}")
    
    # Sort markers by position
    all_markers.sort(key=lambda x: x['position'])
    
    # Save markers to CSV
    markers_df = pd.DataFrame(all_markers)
    markers_df.to_csv(output_csv_path, index=False)
    logging.info(f"Created marker CSV with {len(all_markers)} markers: {output_csv_path}")
    
    return output_wav_path, all_markers, output_csv_path

def calculate_required_segments(required_markers, intro_markers, segment_markers):
    """Calculate how many segment repeats are needed."""
    # Number of markers in intro
    intro_count = len(intro_markers)
    
    # Number of markers in segment
    segment_count = len(segment_markers)
    
    # Calculate needed segments
    remaining = max(0, required_markers - intro_count)
    segments_needed = (remaining + segment_count - 1) // segment_count  # Ceiling division
    
    return segments_needed

def create_concatenated_files(config, required_markers):
    """Create concatenated WAV and CSV files."""
    # Source files
    intro_file = config['paths']['breathing_intro']
    segment_file = config['paths']['breathing_segment']
    outro_file = config['paths']['breathing_outro']
    
    # Check files exist
    if not os.path.exists(intro_file):
        raise FileNotFoundError(f"Intro file not found: {intro_file}")
    if not os.path.exists(segment_file):
        raise FileNotFoundError(f"Segment file not found: {segment_file}")
    
    # Get markers from source files
    intro_markers = extract_markers_from_wav(intro_file)
    segment_markers = extract_markers_from_wav(segment_file)
    
    if not intro_markers:
        raise ValueError(f"No markers found in intro file: {intro_file}")
    if not segment_markers:
        raise ValueError(f"No markers found in segment file: {segment_file}")
    
    # Calculate needed segments
    segments_needed = calculate_required_segments(required_markers, intro_markers, segment_markers)
    
    logging.info(f"Creating concatenated audio for {required_markers} markers:")
    logging.info(f"  Intro markers: {len(intro_markers)}")
    logging.info(f"  Segment markers: {len(segment_markers)}")
    logging.info(f"  Segments needed: {segments_needed}")
    logging.info(f"  Expected markers: ~{len(intro_markers) + (len(segment_markers) * segments_needed)}")
    
    # Build file list
    files_to_concat = [intro_file]
    for _ in range(segments_needed):
        files_to_concat.append(segment_file)
    
    # Add outro if it exists
    if os.path.exists(outro_file):
        files_to_concat.append(outro_file)
    
    # Create output paths
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_wav = os.path.join(
        config['paths']['audio_output_dir'],
        f"BoxBreathing_complete_x{segments_needed}_{timestamp}.wav"
    )
    output_csv = os.path.join(
        config['paths']['audio_output_dir'],
        f"BoxBreathing_complete_x{segments_needed}_{timestamp}_markers.csv"
    )
    
    # Concatenate files and create CSV
    wav_path, markers, csv_path = concatenate_audio_with_markers(
        output_wav,
        output_csv,
        files_to_concat
    )
    
    return wav_path, markers, csv_path

# ----------------------------------------------------------------------
# Main Generator Class
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates PPS experiment designs based on breath markers."""
    
    def __init__(self, config=None):
        """Initialize the generator with configuration."""
        self.config = config or CONFIG
        
        # Prepare directories
        self.exp_log_dir = self.config['paths']['experiment_log_dir']
        self.audio_dir = self.config['paths']['audio_output_dir']
        os.makedirs(self.exp_log_dir, exist_ok=True)
        os.makedirs(self.audio_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Initialize randomization
        self.base_seed = 42
        random.seed(self.base_seed)
        np.random.seed(self.base_seed)
        
        # Create or find audio and extract markers
        self._prepare_audio_and_markers()
    
    def _calculate_trial_counts(self):
        """Calculate number of trials for each condition."""
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        logging.info("Trial count calculation:")
        for cond, count in trial_counts.items():
            logging.info(f"  {cond}: {count} trials")
        logging.info(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _prepare_audio_and_markers(self):
        """Create or find concatenated audio file and markers."""
        # Calculate required markers
        total_trials = sum(self.trial_counts.values())
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = total_trials + practice_trials
        
        # Look for existing files
        existing_files = []
        if os.path.exists(self.audio_dir):
            existing_files = [f for f in os.listdir(self.audio_dir) 
                             if f.startswith("BoxBreathing_complete_") and f.endswith(".wav")]
        
        # Try to use existing file
        self.concat_path = None
        self.markers_csv_path = None
        self.markers = None
        
        if existing_files:
            for file in sorted(existing_files, reverse=True):  # Try newest first
                try:
                    wav_path = os.path.join(self.audio_dir, file)
                    csv_base = os.path.splitext(file)[0]
                    
                    # Look for CSV with markers
                    csv_candidates = [
                        os.path.join(self.audio_dir, f"{csv_base}_markers.csv"),
                        os.path.join(self.audio_dir, f"{csv_base}.csv")
                    ]
                    
                    csv_path = None
                    for candidate in csv_candidates:
                        if os.path.exists(candidate):
                            csv_path = candidate
                            break
                    
                    if csv_path:
                        # Load markers from CSV
                        markers_df = pd.read_csv(csv_path)
                        markers = markers_df.to_dict('records')
                        
                        if len(markers) >= required_markers:
                            self.concat_path = wav_path
                            self.markers_csv_path = csv_path
                            self.markers = markers
                            
                            logging.info(f"Using existing files:")
                            logging.info(f"  WAV: {os.path.basename(wav_path)}")
                            logging.info(f"  CSV: {os.path.basename(csv_path)}")
                            logging.info(f"  Markers: {len(markers)} (need {required_markers})")
                            break
                except Exception as e:
                    logging.warning(f"Error checking existing files: {e}")
        
        # Create new files if needed
        if self.concat_path is None or self.markers_csv_path is None:
            logging.info("Creating new concatenated audio and marker files...")
            self.concat_path, self.markers, self.markers_csv_path = create_concatenated_files(
                self.config, required_markers)
        
        # Load CSV into DataFrame
        self.markers_df = pd.read_csv(self.markers_csv_path)
    
    def generate_counterbalanced_design(self, participant_id):
        """Generate counterbalanced trial conditions."""
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = list(LOOMING_STIMULI.keys())
        repetitions = self.config['repetitions']
        
        all_trials = []
        
        # Generate main trials for each condition
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate trials
            for soa in soa_conditions:
                # Distribute among stimulus types
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle
                random.shuffle(stim_trials)
                
                # Create trials
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True
                    })
        
        # Add catch trials
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly
            stim_counts = {stim: 0 for stim in stimulus_types}
            catch_trials = []
            
            for _ in range(catch_count):
                min_stim = min(stim_counts, key=stim_counts.get)
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                stim_counts[min_stim] += 1
            
            # Add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        return all_trials_df
    
    def assign_trials_to_markers(self, design_df, participant_id):
        """Assign trials to markers based on breathing phase requirements."""
        # Account for practice trials
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = len(design_df) + practice_trials
        
        # Verify we have enough markers
        if len(self.markers) < required_markers:
            raise ValueError(f"Not enough markers ({len(self.markers)}) for required trials ({required_markers})")
        
        # Get copy of markers DataFrame
        markers_df = self.markers_df.copy()
        
        # Sort by position/milliseconds
        if 'milliseconds' in markers_df.columns:
            markers_df = markers_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            markers_df = markers_df.sort_values(by='position').reset_index(drop=True)
        
        # Reserve practice slots
        practice_markers = markers_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_markers = markers_df.iloc[practice_trials:practice_trials+len(design_df)].copy()
        
        # Create practice trials
        practice_trial_list = []
        if practice_trials > 0:
            for i, row in practice_markers.iterrows():
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),
                    'is_tactile': False
                }
                
                # Add all marker columns
                for col in row.index:
                    practice_trial[col] = row[col]
                
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count marker types
        inhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'inhale']
        exhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'exhale']
        n_inhale = len(inhale_markers)
        n_exhale = len(exhale_markers)
        
        logging.info(f"Markers: {n_inhale} inhale, {n_exhale} exhale")
        logging.info(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, "
                    f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough of each type
        if len(pool_inh) > n_inhale:
            raise ValueError(f"Not enough inhale markers ({n_inhale}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale:
            raise ValueError(f"Not enough exhale markers ({n_exhale}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining markers
        remaining_inhale = n_inhale - len(pool_inh)
        remaining_exhale = n_exhale - len(pool_exh)
        total_flex = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough total markers
        if remaining_inhale + remaining_exhale < total_flex:
            raise ValueError(f"Not enough remaining markers ({remaining_inhale + remaining_exhale})"
                            f" for flexible trials ({total_flex})")
        
        # Distribute baseline and catch trials
        baseline_inhale = min(len(pool_baseline) // 2, remaining_inhale)
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = min(len(pool_catch) // 2, remaining_inhale - baseline_inhale)
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create phase pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        logging.info(f"Distributing flexible trials:")
        logging.info(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
        logging.info(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Final pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Map phase to pools
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Assign trials to markers
        regular_assignments = []
        
        for _, row in regular_markers.iterrows():
            phase = row['breathphase'].lower()
            trial_pool = phase_to_pool[phase]
            
            if trial_pool:  # If we still have trials of this phase
                trial = trial_pool.pop(0)  # Get next trial
                
                # Combine trial with marker
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                regular_assignments.append(combined)
        
        # Combine practice and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final DataFrame
        final_df = pd.DataFrame(final_assignments)
        
        # Sort by marker position
        if 'milliseconds' in final_df.columns:
            final_df = final_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            final_df = final_df.sort_values(by='position').reset_index(drop=True)
        
        # Renumber trials
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Add jittered timestamps
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(
            lambda ms: convert_ms_to_timestamp(ms))
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(
            lambda ms: convert_ms_to_timestamp(ms))
        
        return final_df
    
    def finalize_design_csv(self, design_df):
        """Finalize the design CSV with proper formatting."""
        # Set appropriate null values based on trial type
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected_response flag
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        # Determine column order
        marker_cols = ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        trial_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 
                      'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 
                      'soa_ms', 'jittered_ms', 'expected_response']
        
        # Combine and reorder columns
        all_cols = []
        for col in marker_cols:
            if col in design_df.columns:
                all_cols.append(col)
        for col in trial_cols:
            if col in design_df.columns and col not in all_cols:
                all_cols.append(col)
        remaining_cols = [col for col in design_df.columns if col not in all_cols]
        final_order = all_cols + remaining_cols
        
        # Apply column order (only for columns that exist)
        final_order = [col for col in final_order if col in design_df.columns]
        design_df = design_df[final_order]
        
        return design_df
    
    def process_participant(self, participant_id):
        """Process a single participant."""
        try:
            logging.info(f"=== Processing participant {participant_id} ===")
            
            # Generate counterbalanced design
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # Assign to markers
            design_df = self.assign_trials_to_markers(design_df, participant_id)
            
            # Finalize design
            design_df = self.finalize_design_csv(design_df)
            
            # Save to CSV
            design_path = os.path.join(self.exp_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(design_path, index=False)
            
            logging.info(f"Saved design CSV: {design_path}")
            
            return True, {'design_path': design_path}
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """Generate designs for multiple participants."""
        if participant_ids is None:
            participant_ids = list(range(self.config['num_participants']))
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        return results

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def main():
    """Main function."""
    # Print configuration
    print("\n===== PPS DESIGN GENERATOR =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {list(LOOMING_STIMULI.keys())}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    total_trials = total_base_trials + catch_trials + practice_trials
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_trials}")
    
    # Get number of participants from config
    num_participants = CONFIG['num_participants']
    print(f"\nGenerating designs for {num_participants} participants...")
    
    # Create generator and run
    try:
        generator = PPSDesignGenerator(CONFIG)
        results = generator.run()
        
        # Summary
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"\n===== GENERATION SUMMARY =====")
        print(f"Successfully generated {success_count} of {len(results)} participant designs")
        print(f"Audio file: {os.path.basename(generator.concat_path)}")
        print(f"Marker CSV: {os.path.basename(generator.markers_csv_path)}")
        
        print("\nParticipant CSV files:")
        for pid, info in results.items():
            if info['success']:
                print(f"  Participant {pid}: {os.path.basename(info['paths']['design_path'])}")
            else:
                print(f"  Participant {pid}: ERROR - failed to generate")
        
    except Exception as e:
        print(f"Error during design generation: {e}")
        traceback.print_exc()
    
    print("\nDesign generation complete.")
    print("The concatenated audio file and participant designs have been saved.")
    print("Use these files with the audio generator to create the final stimulus files.")

if __name__ == "__main__":
    main()

2025-03-27 20:20:30,536 - INFO - Trial count calculation:
2025-03-27 20:20:30,536 - INFO -   inhalation: 60 trials
2025-03-27 20:20:30,537 - INFO -   exhalation: 60 trials
2025-03-27 20:20:30,537 - INFO -   baseline: 60 trials
2025-03-27 20:20:30,538 - INFO -   catch: 18 trials
2025-03-27 20:20:30,538 - INFO -   Total: 198 trials
2025-03-27 20:20:30,540 - INFO - Creating new concatenated audio and marker files...
2025-03-27 20:20:30,542 - INFO - Found 38 cue points in BoxBreathing_intro_with_markers.wav
2025-03-27 20:20:30,542 - INFO - Extracted 38 markers from BoxBreathing_intro_with_markers.wav
2025-03-27 20:20:30,543 - INFO - First marker: 00:31.9 (inhale)
2025-03-27 20:20:30,545 - INFO - Last marker: 05:28.5 (exhale)
2025-03-27 20:20:30,547 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,548 - INFO - Extracted 34 markers from BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,549 - INFO - First marker: 00:7.1 (inh


===== PPS DESIGN GENERATOR =====
Repetitions (trials per SOA per condition): 12
Practice trials: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206

Generating designs for 100 participants...


2025-03-27 20:20:30,762 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,763 - INFO - Extracted 34 markers from BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,763 - INFO - First marker: 00:7.1 (inhale)
2025-03-27 20:20:30,764 - INFO - Last marker: 04:31.6 (exhale)
2025-03-27 20:20:30,828 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,830 - INFO - Extracted 34 markers from BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,830 - INFO - First marker: 00:7.1 (inhale)
2025-03-27 20:20:30,831 - INFO - Last marker: 04:31.6 (exhale)
2025-03-27 20:20:30,882 - INFO - Found 34 cue points in BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,883 - INFO - Extracted 34 markers from BoxBreathing_repetitionsegment_with_markers.wav
2025-03-27 20:20:30,884 - INFO - First marker: 00:7.1 (inhale)
2025-03-27 20:20:30,885 - INFO - Last marker: 04:31.


===== GENERATION SUMMARY =====
Successfully generated 100 of 100 participant designs
Audio file: BoxBreathing_complete_x5_20250327_202030.wav
Marker CSV: BoxBreathing_complete_x5_20250327_202030_markers.csv

Participant CSV files:
  Participant 0: participant_0_design.csv
  Participant 1: participant_1_design.csv
  Participant 2: participant_2_design.csv
  Participant 3: participant_3_design.csv
  Participant 4: participant_4_design.csv
  Participant 5: participant_5_design.csv
  Participant 6: participant_6_design.csv
  Participant 7: participant_7_design.csv
  Participant 8: participant_8_design.csv
  Participant 9: participant_9_design.csv
  Participant 10: participant_10_design.csv
  Participant 11: participant_11_design.csv
  Participant 12: participant_12_design.csv
  Participant 13: participant_13_design.csv
  Participant 14: participant_14_design.csv
  Participant 15: participant_15_design.csv
  Participant 16: participant_16_design.csv
  Participant 17: participant_17_design.

In [8]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Direct WAV Concatenation with Marker CSV Generation

This script:
1. Creates a concatenated WAV file from source files with markers
2. Directly creates a CSV with calculated marker positions
3. Generates participant-specific design CSVs

The key issue with the previous approach was that markers weren't being preserved
during concatenation. This version manually tracks and calculates marker positions.
"""

import os
import wave
import struct
import numpy as np
import pandas as pd
import random
import traceback
import datetime
from pydub import AudioSegment
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'repetitions': 12,           # Number of trials per SOA condition
    'num_participants': 100,     # Default number of participants
    'practice_trials': 8,        # Number of practice trials at the beginning
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    'paths': {
        # Directory for output files
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'audio_output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        # Input audio files with breath markers
        'breathing_intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro_with_markers.wav",
        'breathing_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment_with_markers.wav",
        'breathing_outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
    },
    'box_breathing': {
        'sample_rate': 48000,         # Audio sample rate (samples per second)
    }
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# ----------------------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------------------
def convert_ms_to_timestamp(ms):
    """Convert milliseconds to timestamp in format MM:SS.S."""
    minutes = int(ms // 60000)
    seconds = (ms % 60000) / 1000.0
    return f"{minutes:02d}:{seconds:.1f}"

def extract_markers_from_wav(wav_file):
    """Extract marker information from a WAV file's cue points."""
    markers = []
    
    try:
        # Get sample rate
        with wave.open(wav_file, 'rb') as wav:
            sample_rate = wav.getframerate()
            frames = wav.getnframes()
            duration_ms = (frames / sample_rate) * 1000
        
        # Open file for binary reading
        with open(wav_file, 'rb') as f:
            # Check RIFF header
            if f.read(4) != b'RIFF':
                raise ValueError("Not a valid RIFF file")
            
            # Skip file size
            f.read(4)
            
            # Check WAVE format
            if f.read(4) != b'WAVE':
                raise ValueError("Not a valid WAVE file")
            
            # Find cue chunk
            while True:
                chunk_pos = f.tell()
                chunk_header = f.read(4)
                
                if not chunk_header or len(chunk_header) < 4:
                    break  # End of file
                
                chunk_size = int.from_bytes(f.read(4), byteorder='little')
                
                if chunk_header == b'cue ':
                    # Found cue chunk
                    num_cues = int.from_bytes(f.read(4), byteorder='little')
                    logging.info(f"Found {num_cues} cue points in {os.path.basename(wav_file)}")
                    
                    # Read all cue points
                    for i in range(num_cues):
                        # Read cue ID and position
                        cue_id = int.from_bytes(f.read(4), byteorder='little')
                        position = int.from_bytes(f.read(4), byteorder='little')
                        
                        # Skip remaining cue point data
                        f.read(16)
                        
                        # Calculate timestamp
                        position_ms = (position / sample_rate) * 1000
                        
                        # Determine breathphase from ID parity
                        breathphase = "inhale" if cue_id % 2 == 1 else "exhale"
                        
                        markers.append({
                            'id': cue_id,
                            'position': position,
                            'milliseconds': position_ms,
                            'retentionslot_timestamp': convert_ms_to_timestamp(position_ms),
                            'breathphase': breathphase,
                            'description': f"{breathphase.capitalize()} hold marker"
                        })
                    
                    break
                else:
                    # Skip to next chunk
                    f.seek(chunk_pos + 8 + chunk_size)
        
        # Sort markers by position
        markers.sort(key=lambda x: x['position'])
        
        if markers:
            logging.info(f"Extracted {len(markers)} markers from {os.path.basename(wav_file)}")
            logging.info(f"First marker: {markers[0]['retentionslot_timestamp']} ({markers[0]['breathphase']})")
            logging.info(f"Last marker: {markers[-1]['retentionslot_timestamp']} ({markers[-1]['breathphase']})")
        else:
            logging.warning(f"No markers found in {os.path.basename(wav_file)}")
        
    except Exception as e:
        logging.error(f"Error extracting markers from {os.path.basename(wav_file)}: {e}")
        traceback.print_exc()
    
    return markers

def concatenate_audio_with_markers(output_wav_path, output_csv_path, files_to_concat):
    """
    Concatenate audio files and manually create a marker CSV.
    
    Args:
        output_wav_path: Path to save the concatenated WAV file
        output_csv_path: Path to save the marker CSV file
        files_to_concat: List of WAV files to concatenate
    
    Returns:
        Tuple of (WAV path, marker list, CSV path)
    """
    # Check input files
    valid_files = [f for f in files_to_concat if os.path.exists(f)]
    if not valid_files:
        raise ValueError("No valid audio files to concatenate")
    
    # Create output directory if needed
    os.makedirs(os.path.dirname(output_wav_path), exist_ok=True)
    
    # Extract markers from each file
    all_markers = []
    audio_segments = []
    current_offset_ms = 0
    
    for file_path in valid_files:
        # Load audio segment
        audio = AudioSegment.from_wav(file_path)
        audio_segments.append(audio)
        
        # Get markers and adjust positions
        markers = extract_markers_from_wav(file_path)
        
        # Adjust marker positions based on offset
        for marker in markers:
            marker['milliseconds'] += current_offset_ms
            marker['retentionslot_timestamp'] = convert_ms_to_timestamp(marker['milliseconds'])
            marker['position'] += int(current_offset_ms * audio.frame_rate / 1000)
            marker['sample_index'] = marker['position']  # Add sample_index for compatibility
        
        # Add adjusted markers
        all_markers.extend(markers)
        
        # Update offset for next file
        current_offset_ms += len(audio)
    
    # Concatenate audio
    concatenated_audio = audio_segments[0]
    for segment in audio_segments[1:]:
        concatenated_audio += segment
    
    # Save concatenated WAV
    concatenated_audio.export(output_wav_path, format="wav")
    logging.info(f"Created concatenated audio: {output_wav_path}")
    
    # Sort markers by position
    all_markers.sort(key=lambda x: x['position'])
    
    # Save markers to CSV
    markers_df = pd.DataFrame(all_markers)
    markers_df.to_csv(output_csv_path, index=False)
    logging.info(f"Created marker CSV with {len(all_markers)} markers: {output_csv_path}")
    
    return output_wav_path, all_markers, output_csv_path

def calculate_required_segments(required_markers, intro_markers, segment_markers):
    """Calculate how many segment repeats are needed."""
    # Number of markers in intro
    intro_count = len(intro_markers)
    
    # Number of markers in segment
    segment_count = len(segment_markers)
    
    # Calculate needed segments
    remaining = max(0, required_markers - intro_count)
    segments_needed = (remaining + segment_count - 1) // segment_count  # Ceiling division
    
    return segments_needed

def create_concatenated_files(config, required_markers):
    """Create concatenated WAV and CSV files."""
    # Source files
    intro_file = config['paths']['breathing_intro']
    segment_file = config['paths']['breathing_segment']
    outro_file = config['paths']['breathing_outro']
    
    # Check files exist
    if not os.path.exists(intro_file):
        raise FileNotFoundError(f"Intro file not found: {intro_file}")
    if not os.path.exists(segment_file):
        raise FileNotFoundError(f"Segment file not found: {segment_file}")
    
    # Get markers from source files
    intro_markers = extract_markers_from_wav(intro_file)
    segment_markers = extract_markers_from_wav(segment_file)
    
    if not intro_markers:
        raise ValueError(f"No markers found in intro file: {intro_file}")
    if not segment_markers:
        raise ValueError(f"No markers found in segment file: {segment_file}")
    
    # Calculate needed segments
    segments_needed = calculate_required_segments(required_markers, intro_markers, segment_markers)
    
    logging.info(f"Creating concatenated audio for {required_markers} markers:")
    logging.info(f"  Intro markers: {len(intro_markers)}")
    logging.info(f"  Segment markers: {len(segment_markers)}")
    logging.info(f"  Segments needed: {segments_needed}")
    logging.info(f"  Expected markers: ~{len(intro_markers) + (len(segment_markers) * segments_needed)}")
    
    # Build file list
    files_to_concat = [intro_file]
    for _ in range(segments_needed):
        files_to_concat.append(segment_file)
    
    # Add outro if it exists
    if os.path.exists(outro_file):
        files_to_concat.append(outro_file)
    
    # Create output paths
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_wav = os.path.join(
        config['paths']['audio_output_dir'],
        f"BoxBreathing_complete_x{segments_needed}_{timestamp}.wav"
    )
    output_csv = os.path.join(
        config['paths']['audio_output_dir'],
        f"BoxBreathing_complete_x{segments_needed}_{timestamp}_markers.csv"
    )
    
    # Concatenate files and create CSV
    wav_path, markers, csv_path = concatenate_audio_with_markers(
        output_wav,
        output_csv,
        files_to_concat
    )
    
    return wav_path, markers, csv_path

# ----------------------------------------------------------------------
# Main Generator Class
# ----------------------------------------------------------------------
class PPSDesignGenerator:
    """Generates PPS experiment designs based on breath markers."""
    
    def __init__(self, config=None):
        """Initialize the generator with configuration."""
        self.config = config or CONFIG
        
        # Prepare directories
        self.exp_log_dir = self.config['paths']['experiment_log_dir']
        self.audio_dir = self.config['paths']['audio_output_dir']
        os.makedirs(self.exp_log_dir, exist_ok=True)
        os.makedirs(self.audio_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Initialize randomization
        self.base_seed = 42
        random.seed(self.base_seed)
        np.random.seed(self.base_seed)
        
        # Create or find audio and extract markers
        self._prepare_audio_and_markers()
    
    def _calculate_trial_counts(self):
        """Calculate number of trials for each condition."""
        soa_conditions = self.config['soa_conditions_ms']
        repetitions = self.config['repetitions']
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # Each condition gets repetitions × number of SOAs trials
                trial_counts[condition] = len(soa_conditions) * repetitions
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        logging.info("Trial count calculation:")
        for cond, count in trial_counts.items():
            logging.info(f"  {cond}: {count} trials")
        logging.info(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _prepare_audio_and_markers(self):
        """Create or find concatenated audio file and markers."""
        # Calculate required markers
        total_trials = sum(self.trial_counts.values())
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = total_trials + practice_trials
        
        # Look for existing files
        existing_files = []
        if os.path.exists(self.audio_dir):
            existing_files = [f for f in os.listdir(self.audio_dir) 
                             if f.startswith("BoxBreathing_complete_") and f.endswith(".wav")]
        
        # Try to use existing file
        self.concat_path = None
        self.markers_csv_path = None
        self.markers = None
        
        if existing_files:
            for file in sorted(existing_files, reverse=True):  # Try newest first
                try:
                    wav_path = os.path.join(self.audio_dir, file)
                    csv_base = os.path.splitext(file)[0]
                    
                    # Look for CSV with markers
                    csv_candidates = [
                        os.path.join(self.audio_dir, f"{csv_base}_markers.csv"),
                        os.path.join(self.audio_dir, f"{csv_base}.csv")
                    ]
                    
                    csv_path = None
                    for candidate in csv_candidates:
                        if os.path.exists(candidate):
                            csv_path = candidate
                            break
                    
                    if csv_path:
                        # Load markers from CSV
                        markers_df = pd.read_csv(csv_path)
                        markers = markers_df.to_dict('records')
                        
                        if len(markers) >= required_markers:
                            self.concat_path = wav_path
                            self.markers_csv_path = csv_path
                            self.markers = markers
                            
                            logging.info(f"Using existing files:")
                            logging.info(f"  WAV: {os.path.basename(wav_path)}")
                            logging.info(f"  CSV: {os.path.basename(csv_path)}")
                            logging.info(f"  Markers: {len(markers)} (need {required_markers})")
                            break
                except Exception as e:
                    logging.warning(f"Error checking existing files: {e}")
        
        # Create new files if needed
        if self.concat_path is None or self.markers_csv_path is None:
            logging.info("Creating new concatenated audio and marker files...")
            self.concat_path, self.markers, self.markers_csv_path = create_concatenated_files(
                self.config, required_markers)
        
        # Load CSV into DataFrame
        self.markers_df = pd.read_csv(self.markers_csv_path)
    
    def generate_counterbalanced_design(self, participant_id):
        """Generate counterbalanced trial conditions."""
        random.seed(self.base_seed + participant_id)
        np.random.seed(self.base_seed + participant_id)
        
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = list(LOOMING_STIMULI.keys())
        repetitions = self.config['repetitions']
        
        all_trials = []
        
        # Generate main trials for each condition
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # For each SOA, generate trials
            for soa in soa_conditions:
                # Distribute among stimulus types
                trials_per_soa = repetitions
                trials_per_stim = trials_per_soa // len(stimulus_types)
                remainder = trials_per_soa % len(stimulus_types)
                
                # Distribute stimulus types
                stim_trials = []
                for i, stim_type in enumerate(stimulus_types):
                    extra = 1 if i < remainder else 0
                    for _ in range(trials_per_stim + extra):
                        stim_trials.append(stim_type)
                
                # Shuffle
                random.shuffle(stim_trials)
                
                # Create trials
                for stim_type in stim_trials:
                    all_trials.append({
                        'participant_id': participant_id,
                        'trial_type': condition,
                        'stimulus_type': stim_type,
                        'soa_value_ms': soa,
                        'jitter_ms': random.choice(jitter_options),
                        'is_tactile': True
                    })
        
        # Add catch trials
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            # Distribute catch trials evenly
            stim_counts = {stim: 0 for stim in stimulus_types}
            catch_trials = []
            
            for _ in range(catch_count):
                min_stim = min(stim_counts, key=stim_counts.get)
                catch_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': min_stim,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
                stim_counts[min_stim] += 1
            
            # Add to all trials
            random.shuffle(catch_trials)
            all_trials.extend(catch_trials)
        
        # Shuffle and assign trial numbers
        all_trials_df = pd.DataFrame(all_trials)
        all_trials_df = all_trials_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        all_trials_df['trial_number'] = range(1, len(all_trials_df) + 1)
        
        return all_trials_df
    
    def assign_trials_to_markers(self, design_df, participant_id):
        """Assign trials to markers based on breathing phase requirements."""
        # Account for practice trials
        practice_trials = self.config.get('practice_trials', 0)
        required_markers = len(design_df) + practice_trials
        
        # Verify we have enough markers
        if len(self.markers) < required_markers:
            raise ValueError(f"Not enough markers ({len(self.markers)}) for required trials ({required_markers})")
        
        # Get copy of markers DataFrame
        markers_df = self.markers_df.copy()
        
        # Sort by position/milliseconds
        if 'milliseconds' in markers_df.columns:
            markers_df = markers_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            markers_df = markers_df.sort_values(by='position').reset_index(drop=True)
        
        # Reserve practice slots
        practice_markers = markers_df.iloc[:practice_trials].copy() if practice_trials > 0 else pd.DataFrame()
        regular_markers = markers_df.iloc[practice_trials:practice_trials+len(design_df)].copy()
        
        # Create practice trials
        practice_trial_list = []
        if practice_trials > 0:
            for i, row in practice_markers.iterrows():
                practice_trial = {
                    'participant_id': participant_id,
                    'trial_number': i + 1,
                    'trial_type': 'practice',
                    'stimulus_type': None,
                    'soa_value_ms': None,
                    'jitter_ms': random.choice(self.config['jitter_options_ms']),
                    'is_tactile': False
                }
                
                # Add all marker columns
                for col in row.index:
                    practice_trial[col] = row[col]
                
                practice_trial_list.append(practice_trial)
        
        # Partition design trials by type
        pool_inh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'inhalation']
        pool_exh = [d for d in design_df.to_dict('records') if d['trial_type'] == 'exhalation']
        pool_baseline = [d for d in design_df.to_dict('records') if d['trial_type'] == 'baseline']
        pool_catch = [d for d in design_df.to_dict('records') if d['trial_type'] == 'catch']
        
        # Count marker types
        inhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'inhale']
        exhale_markers = regular_markers[regular_markers['breathphase'].str.lower() == 'exhale']
        n_inhale = len(inhale_markers)
        n_exhale = len(exhale_markers)
        
        logging.info(f"Markers: {n_inhale} inhale, {n_exhale} exhale")
        logging.info(f"Trials: {len(pool_inh)} inhalation, {len(pool_exh)} exhalation, "
                    f"{len(pool_baseline)} baseline, {len(pool_catch)} catch")
        
        # Verify we have enough of each type
        if len(pool_inh) > n_inhale:
            raise ValueError(f"Not enough inhale markers ({n_inhale}) for inhalation trials ({len(pool_inh)})")
        if len(pool_exh) > n_exhale:
            raise ValueError(f"Not enough exhale markers ({n_exhale}) for exhalation trials ({len(pool_exh)})")
        
        # Calculate remaining markers
        remaining_inhale = n_inhale - len(pool_inh)
        remaining_exhale = n_exhale - len(pool_exh)
        total_flex = len(pool_baseline) + len(pool_catch)
        
        # Verify we have enough total markers
        if remaining_inhale + remaining_exhale < total_flex:
            raise ValueError(f"Not enough remaining markers ({remaining_inhale + remaining_exhale})"
                            f" for flexible trials ({total_flex})")
        
        # Distribute baseline and catch trials
        baseline_inhale = min(len(pool_baseline) // 2, remaining_inhale)
        baseline_exhale = len(pool_baseline) - baseline_inhale
        
        catch_inhale = min(len(pool_catch) // 2, remaining_inhale - baseline_inhale)
        catch_exhale = len(pool_catch) - catch_inhale
        
        # Create phase pools
        inhale_flexible = pool_baseline[:baseline_inhale] + pool_catch[:catch_inhale]
        exhale_flexible = pool_baseline[baseline_inhale:] + pool_catch[catch_inhale:]
        
        random.shuffle(inhale_flexible)
        random.shuffle(exhale_flexible)
        
        logging.info(f"Distributing flexible trials:")
        logging.info(f"  Baseline: {baseline_inhale} to inhale, {baseline_exhale} to exhale")
        logging.info(f"  Catch: {catch_inhale} to inhale, {catch_exhale} to exhale")
        
        # Final pools
        inhale_pool = pool_inh + inhale_flexible
        exhale_pool = pool_exh + exhale_flexible
        
        random.shuffle(inhale_pool)
        random.shuffle(exhale_pool)
        
        # Map phase to pools
        phase_to_pool = {
            'inhale': inhale_pool,
            'exhale': exhale_pool
        }
        
        # Assign trials to markers
        regular_assignments = []
        
        for _, row in regular_markers.iterrows():
            phase = row['breathphase'].lower()
            trial_pool = phase_to_pool[phase]
            
            if trial_pool:  # If we still have trials of this phase
                trial = trial_pool.pop(0)  # Get next trial
                
                # Combine trial with marker
                combined = trial.copy()
                for col in row.index:
                    combined[col] = row[col]
                
                regular_assignments.append(combined)
        
        # Combine practice and regular trials
        final_assignments = practice_trial_list + regular_assignments
        
        # Create final DataFrame
        final_df = pd.DataFrame(final_assignments)
        
        # Sort by marker position
        if 'milliseconds' in final_df.columns:
            final_df = final_df.sort_values(by='milliseconds').reset_index(drop=True)
        else:
            final_df = final_df.sort_values(by='position').reset_index(drop=True)
        
        # Renumber trials
        final_df['trial_number'] = range(1, len(final_df) + 1)
        
        # Add jittered timestamps
        final_df['jittered_ms'] = final_df['milliseconds'] + final_df['jitter_ms'].fillna(0)
        
        # Add looming stimulus timestamp for non-baseline trials
        looming_mask = ~final_df['trial_type'].isin(['baseline', 'practice'])
        final_df.loc[looming_mask, 'looming_stimulus_timestamp'] = final_df.loc[looming_mask, 'jittered_ms'].apply(
            lambda ms: convert_ms_to_timestamp(ms))
        
        # Add SOA timing for non-catch, non-practice trials
        non_catch_mask = ~final_df['trial_type'].isin(['catch', 'practice'])
        final_df.loc[non_catch_mask, 'soa_ms'] = final_df.loc[non_catch_mask, 'jittered_ms'] + final_df.loc[non_catch_mask, 'soa_value_ms'].fillna(0)
        final_df.loc[non_catch_mask, 'tactile_stimulus_timestamp'] = final_df.loc[non_catch_mask, 'soa_ms'].apply(
            lambda ms: convert_ms_to_timestamp(ms))
        
        return final_df
    
    def finalize_design_csv(self, design_df):
        """Finalize the design CSV with proper formatting."""
        # Set appropriate null values based on trial type
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'stimulus_type'] = None
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = None
        
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = None
        
        practice_mask = (design_df['trial_type'] == 'practice')
        design_df.loc[practice_mask, 'stimulus_type'] = None
        design_df.loc[practice_mask, 'looming_stimulus_timestamp'] = None
        design_df.loc[practice_mask, 'tactile_stimulus_timestamp'] = None
        
        # Add expected_response flag
        design_df['expected_response'] = ~design_df['trial_type'].isin(['catch', 'practice'])
        
        # Determine column order
        marker_cols = ['retentionslot_timestamp', 'milliseconds', 'sample_index', 'description', 'breathphase']
        trial_cols = ['participant_id', 'trial_number', 'trial_type', 'stimulus_type', 'soa_value_ms', 
                      'jitter_ms', 'is_tactile', 'looming_stimulus_timestamp', 'tactile_stimulus_timestamp', 
                      'soa_ms', 'jittered_ms', 'expected_response']
        
        # Combine and reorder columns
        all_cols = []
        for col in marker_cols:
            if col in design_df.columns:
                all_cols.append(col)
        for col in trial_cols:
            if col in design_df.columns and col not in all_cols:
                all_cols.append(col)
        remaining_cols = [col for col in design_df.columns if col not in all_cols]
        final_order = all_cols + remaining_cols
        
        # Apply column order (only for columns that exist)
        final_order = [col for col in final_order if col in design_df.columns]
        design_df = design_df[final_order]
        
        return design_df
    
    def process_participant(self, participant_id):
        """Process a single participant."""
        try:
            logging.info(f"=== Processing participant {participant_id} ===")
            
            # Generate counterbalanced design
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # Assign to markers
            design_df = self.assign_trials_to_markers(design_df, participant_id)
            
            # Finalize design
            design_df = self.finalize_design_csv(design_df)
            
            # Save to CSV
            design_path = os.path.join(self.exp_log_dir, f"participant_{participant_id}_design.csv")
            design_df.to_csv(design_path, index=False)
            
            logging.info(f"Saved design CSV: {design_path}")
            
            return True, {'design_path': design_path}
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """Generate designs for multiple participants."""
        if participant_ids is None:
            participant_ids = list(range(self.config['num_participants']))
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        return results

# ----------------------------------------------------------------------
# Main Function
# ----------------------------------------------------------------------
def main():
    """Main function."""
    # Print configuration
    print("\n===== PPS DESIGN GENERATOR =====")
    print(f"Repetitions (trials per SOA per condition): {CONFIG['repetitions']}")
    print(f"Practice trials: {CONFIG['practice_trials']}")
    print(f"SOA Conditions: {CONFIG['soa_conditions_ms']}")
    print(f"Jitter Options: {CONFIG['jitter_options_ms']}")
    print(f"Looming Stimuli: {list(LOOMING_STIMULI.keys())}")
    
    # Calculate expected trial counts
    reps = CONFIG['repetitions']
    soa_count = len(CONFIG['soa_conditions_ms'])
    trials_per_condition = reps * soa_count
    enabled_conditions = [cond for cond, enabled in CONFIG['included_conditions'].items() if enabled]
    total_base_trials = trials_per_condition * len(enabled_conditions)
    catch_trials = int(np.ceil(total_base_trials * CONFIG['catch_trial_percentage'] / 100))
    practice_trials = CONFIG['practice_trials']
    total_trials = total_base_trials + catch_trials + practice_trials
    
    print("\n===== EXPECTED TRIAL COUNTS =====")
    print(f"Trials per condition: {reps} trials per SOA × {soa_count} SOAs = {trials_per_condition}")
    for cond in enabled_conditions:
        print(f"  {cond.upper()}: {trials_per_condition} trials")
    print(f"Catch trials ({CONFIG['catch_trial_percentage']}%): {catch_trials}")
    print(f"Practice trials: {practice_trials}")
    print(f"TOTAL TRIALS PER PARTICIPANT: {total_trials}")
    
    # Get number of participants from config
    num_participants = CONFIG['num_participants']
    print(f"\nGenerating designs for {num_participants} participants...")
    
    # Create generator and run
    try:
        generator = PPSDesignGenerator(CONFIG)
        results = generator.run()
        
        # Summary
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"\n===== GENERATION SUMMARY =====")
        print(f"Successfully generated {success_count} of {len(results)} participant designs")
        print(f"Audio file: {os.path.basename(generator.concat_path)}")
        print(f"Marker CSV: {os.path.basename(generator.markers_csv_path)}")
        
        print("\nParticipant CSV files:")
        for pid, info in results.items():
            if info['success']:
                print(f"  Participant {pid}: {os.path.basename(info['paths']['design_path'])}")
            else:
                print(f"  Participant {pid}: ERROR - failed to generate")
        
    except Exception as e:
        print(f"Error during design generation: {e}")
        traceback.print_exc()
    
    print("\nDesign generation complete.")
    print("The concatenated audio file and participant designs have been saved.")
    print("Use these files with the audio generator to create the final stimulus files.")

if __name__ == "__main__":
    main()

2025-03-27 20:35:34,039 - INFO - Trial count calculation:
2025-03-27 20:35:34,040 - INFO -   inhalation: 60 trials
2025-03-27 20:35:34,041 - INFO -   exhalation: 60 trials
2025-03-27 20:35:34,041 - INFO -   baseline: 60 trials
2025-03-27 20:35:34,041 - INFO -   catch: 18 trials
2025-03-27 20:35:34,043 - INFO -   Total: 198 trials
2025-03-27 20:35:34,050 - INFO - Using existing files:
2025-03-27 20:35:34,051 - INFO -   WAV: BoxBreathing_complete_x5_20250327_202030.wav
2025-03-27 20:35:34,051 - INFO -   CSV: BoxBreathing_complete_x5_20250327_202030_markers.csv
2025-03-27 20:35:34,051 - INFO -   Markers: 208 (need 206)
2025-03-27 20:35:34,055 - INFO - === Processing participant 0 ===
2025-03-27 20:35:34,066 - INFO - Markers: 99 inhale, 99 exhale
2025-03-27 20:35:34,066 - INFO - Trials: 60 inhalation, 60 exhalation, 60 baseline, 18 catch
2025-03-27 20:35:34,066 - INFO - Distributing flexible trials:
2025-03-27 20:35:34,066 - INFO -   Baseline: 30 to inhale, 30 to exhale
2025-03-27 20:35:34


===== PPS DESIGN GENERATOR =====
Repetitions (trials per SOA per condition): 12
Practice trials: 8
SOA Conditions: [190, 400, 700, 1000, 1500]
Jitter Options: [100, 200, 300, 400, 500]
Looming Stimuli: ['blue', 'brown', 'pink', 'white']

===== EXPECTED TRIAL COUNTS =====
Trials per condition: 12 trials per SOA × 5 SOAs = 60
  INHALATION: 60 trials
  EXHALATION: 60 trials
  BASELINE: 60 trials
Catch trials (10%): 18
Practice trials: 8
TOTAL TRIALS PER PARTICIPANT: 206

Generating designs for 100 participants...


2025-03-27 20:35:34,247 - INFO - === Processing participant 3 ===
2025-03-27 20:35:34,254 - INFO - Markers: 99 inhale, 99 exhale
2025-03-27 20:35:34,254 - INFO - Trials: 60 inhalation, 60 exhalation, 60 baseline, 18 catch
2025-03-27 20:35:34,254 - INFO - Distributing flexible trials:
2025-03-27 20:35:34,260 - INFO -   Baseline: 30 to inhale, 30 to exhale
2025-03-27 20:35:34,260 - INFO -   Catch: 9 to inhale, 9 to exhale
2025-03-27 20:35:34,299 - INFO - Saved design CSV: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_3_design.csv
2025-03-27 20:35:34,302 - INFO - === Processing participant 4 ===
2025-03-27 20:35:34,312 - INFO - Markers: 99 inhale, 99 exhale
2025-03-27 20:35:34,312 - INFO - Trials: 60 inhalation, 60 exhalation, 60 baseline, 18 catch
2025-03-27 20:35:34,312 - INFO - Distributing flexible trials:
2025-03-27 20:35:34,312 - INFO -   Baseline: 30 to inhale, 30 to exhale
2025-03-27 20:35:34,312 - INFO -   Catch: 9 to inhal


===== GENERATION SUMMARY =====
Successfully generated 100 of 100 participant designs
Audio file: BoxBreathing_complete_x5_20250327_202030.wav
Marker CSV: BoxBreathing_complete_x5_20250327_202030_markers.csv

Participant CSV files:
  Participant 0: participant_0_design.csv
  Participant 1: participant_1_design.csv
  Participant 2: participant_2_design.csv
  Participant 3: participant_3_design.csv
  Participant 4: participant_4_design.csv
  Participant 5: participant_5_design.csv
  Participant 6: participant_6_design.csv
  Participant 7: participant_7_design.csv
  Participant 8: participant_8_design.csv
  Participant 9: participant_9_design.csv
  Participant 10: participant_10_design.csv
  Participant 11: participant_11_design.csv
  Participant 12: participant_12_design.csv
  Participant 13: participant_13_design.csv
  Participant 14: participant_14_design.csv
  Participant 15: participant_15_design.csv
  Participant 16: participant_16_design.csv
  Participant 17: participant_17_design.

In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Audio Generator

This script generates participant-specific audio files for a Peripersonal Space (PPS) experiment.
It reads design CSV files from the ExperimentLog directory and creates:
1. A looming audio file with the parent audio and looming stimuli overlaid at specified timestamps
2. A tactile audio file with tactile stimuli at specified timestamps

Author: Claude
"""

import os
import numpy as np
import pandas as pd
import soundfile as sf
import re
import logging
from pathlib import Path
import traceback

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration
CONFIG = {
    'paths': {
        # Find the most recent parent audio file with "BoxBreathing_complete" prefix
        'design_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles",
    },
    'debug_mode': True,
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# Helper function for parsing timestamps
def parse_timestamp(timestamp):
    """
    Parse timestamp in format MM:SS.S to milliseconds.
    
    Args:
        timestamp: String in format "MM:SS.S"
        
    Returns:
        Float: milliseconds
    """
    if pd.isna(timestamp):
        return None
        
    match = re.match(r'(\d+):(\d+\.\d+)', timestamp)
    if match:
        minutes, seconds = match.groups()
        return (int(minutes) * 60 + float(seconds)) * 1000
    return None

class PPSAudioGenerator:
    """Generates participant-specific audio files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.output_dir = self.config['paths']['output_dir']
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Find parent audio file (most recent one with BoxBreathing_complete prefix)
        self.parent_audio_path = self._find_parent_audio()
        if not self.parent_audio_path:
            raise FileNotFoundError("Could not find parent audio file with BoxBreathing_complete prefix")
        
        # Load parent audio
        self.parent_audio, self.sample_rate = self._load_audio(self.parent_audio_path)
        
        # Load stimulus files
        self.looming_stimuli = self._load_looming_stimuli()
        self.tactile_stimulus, _ = self._load_audio(os.path.join(
            self.config['paths']['stimulus_dir'], TACTILE_STIMULUS))
        
        if self.config['debug_mode']:
            self._log_audio_info()
    
    def _find_parent_audio(self):
        """Find the most recent parent audio file in the output directory."""
        audio_dir = self.config['paths']['output_dir']
        parent_files = [f for f in os.listdir(audio_dir) 
                      if f.startswith("BoxBreathing_complete") and f.endswith(".wav")]
        
        if not parent_files:
            return None
        
        # Sort by creation time (most recent first)
        parent_files.sort(key=lambda f: os.path.getmtime(os.path.join(audio_dir, f)), reverse=True)
        
        return os.path.join(audio_dir, parent_files[0])
    
    def _load_audio(self, file_path):
        """Load audio file and return data and sample rate."""
        try:
            data, sample_rate = sf.read(file_path)
            return data, sample_rate
        except Exception as e:
            logging.error(f"Error loading audio file {file_path}: {e}")
            raise
    
    def _load_looming_stimuli(self):
        """Load all looming stimulus files."""
        looming_stimuli = {}
        for key, filename in LOOMING_STIMULI.items():
            file_path = os.path.join(self.config['paths']['stimulus_dir'], filename)
            try:
                data, _ = self._load_audio(file_path)
                looming_stimuli[key] = data
                if self.config['debug_mode']:
                    logging.info(f"Loaded looming stimulus '{key}' from {filename}")
            except Exception as e:
                logging.warning(f"Could not load looming stimulus '{key}' from {filename}: {e}")
        return looming_stimuli
    
    def _log_audio_info(self):
        """Log information about loaded audio files for debugging."""
        logging.info(f"Loaded parent audio: {self.parent_audio_path}")
        logging.info(f"  Duration: {len(self.parent_audio)/self.sample_rate:.2f} seconds")
        logging.info(f"  Sample Rate: {self.sample_rate} Hz")
        logging.info(f"  Shape: {self.parent_audio.shape}")
        
        for key, data in self.looming_stimuli.items():
            logging.info(f"  Looming '{key}': {len(data)/self.sample_rate:.2f} seconds, {data.shape}")
        
        logging.info(f"  Tactile: {len(self.tactile_stimulus)/self.sample_rate:.2f} seconds, {self.tactile_stimulus.shape}")
        logging.info("=== TIMESTAMP HIERARCHY ===")
        logging.info("Required ordering: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp")
        logging.info("Validating this ordering before generating audio files")
    
    def load_participant_design(self, participant_id):
        """
        Load participant design CSV and parse timestamps.
        
        Args:
            participant_id: Participant ID number
            
        Returns:
            DataFrame containing the trial design
        """
        design_path = os.path.join(
            self.config['paths']['design_dir'], 
            f"participant_{participant_id}_design.csv")
        
        try:
            design_df = pd.read_csv(design_path)
            
            # Parse timestamp columns to milliseconds
            for col in ['looming_stimulus_timestamp', 'tactile_stimulus_timestamp']:
                if col in design_df.columns:
                    design_df[f'{col}_ms'] = design_df[col].apply(parse_timestamp)
            
            if self.config['debug_mode']:
                max_trial = design_df['trial_number'].max() if 'trial_number' in design_df.columns else len(design_df)
                logging.info(f"Loaded design for participant {participant_id}")
                logging.info(f"  Total trials: {len(design_df)}")
                logging.info(f"  Max trial number: {max_trial}")
                if 'trial_type' in design_df.columns:
                    logging.info(f"  Trial types: {design_df['trial_type'].value_counts().to_dict()}")
            
            return design_df
            
        except Exception as e:
            logging.error(f"Error loading design for participant {participant_id}: {e}")
            raise
    
    def create_injection_map(self, design_df):
        """
        Create a map of where to inject looming and tactile stimuli.
        Validates timestamp hierarchy: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp
        
        Args:
            design_df: DataFrame with the design
            
        Returns:
            Dictionary with injection details and validation results
        """
        injection_map = {
            'looming': [],
            'tactile': []
        }
        
        # Track validation issues
        validation_issues = []
        
        # Process each trial
        for _, row in design_df.iterrows():
            # Skip if no trial_type column or if it's a practice trial
            if 'trial_type' not in row or row['trial_type'] == 'practice':
                continue
            
            trial_type = row['trial_type']
            trial_number = row.get('trial_number', 0)
            
            # Get retention timestamp (should be earliest)
            retention_ms = None
            if 'retentionslot_timestamp' in row and not pd.isna(row['retentionslot_timestamp']):
                retention_ms = parse_timestamp(row['retentionslot_timestamp'])
                
            # Get looming timestamp
            looming_ms = None
            if trial_type in ['inhalation', 'exhalation', 'catch'] and 'looming_stimulus_timestamp' in row and not pd.isna(row['looming_stimulus_timestamp']):
                looming_ms = parse_timestamp(row['looming_stimulus_timestamp'])
                
                # Validate looming comes after retention
                if retention_ms is not None and looming_ms is not None and looming_ms <= retention_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Looming timestamp ({looming_ms} ms) should be after retention timestamp ({retention_ms} ms)",
                        'retention_ts': row.get('retentionslot_timestamp', 'N/A'),
                        'looming_ts': row.get('looming_stimulus_timestamp', 'N/A')
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                    
                if looming_ms is not None:
                    stim_type = row['stimulus_type'] if 'stimulus_type' in row and not pd.isna(row['stimulus_type']) else 'white'
                    
                    if stim_type in self.looming_stimuli:
                        injection_map['looming'].append({
                            'time_ms': looming_ms,
                            'timestamp': row['looming_stimulus_timestamp'],
                            'stimulus_type': stim_type,
                            'trial_number': trial_number,
                            'trial_type': trial_type,
                            'retention_ms': retention_ms
                        })
            
            # Get tactile timestamp
            tactile_ms = None
            if trial_type in ['inhalation', 'exhalation', 'baseline'] and 'tactile_stimulus_timestamp' in row and not pd.isna(row['tactile_stimulus_timestamp']):
                tactile_ms = parse_timestamp(row['tactile_stimulus_timestamp'])
                
                # Validate tactile comes after looming (if applicable)
                if looming_ms is not None and tactile_ms is not None and tactile_ms <= looming_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Tactile timestamp ({tactile_ms} ms) should be after looming timestamp ({looming_ms} ms)",
                        'looming_ts': row.get('looming_stimulus_timestamp', 'N/A'),
                        'tactile_ts': row.get('tactile_stimulus_timestamp', 'N/A')
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                
                # For baseline trials, validate tactile comes after retention
                if trial_type == 'baseline' and retention_ms is not None and tactile_ms is not None and tactile_ms <= retention_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Baseline tactile timestamp ({tactile_ms} ms) should be after retention timestamp ({retention_ms} ms)",
                        'retention_ts': row.get('retentionslot_timestamp', 'N/A'),
                        'tactile_ts': row.get('tactile_stimulus_timestamp', 'N/A')
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                    
                if tactile_ms is not None:
                    injection_map['tactile'].append({
                        'time_ms': tactile_ms,
                        'timestamp': row['tactile_stimulus_timestamp'],
                        'trial_number': trial_number,
                        'trial_type': trial_type,
                        'looming_ms': looming_ms,
                        'retention_ms': retention_ms
                    })
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
            if validation_issues:
                logging.warning(f"  Found {len(validation_issues)} timestamp validation issues!")
            else:
                logging.info("  All timestamps validated successfully (retentionslot < looming < tactile)")
        
        return injection_map, validation_issues
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
        
        return injection_map
    
    def create_participant_audio(self, participant_id, injection_map):
        """
        Create participant-specific audio by overlaying stimuli on parent audio.
        
        Args:
            participant_id: Participant ID
            injection_map: Dictionary with injection details
            
        Returns:
            Tuple of (looming_audio, tactile_audio, injection_log)
        """
        # Create copies of parent audio for looming
        looming_audio = self.parent_audio.copy()
        
        # Create empty audio of same length for tactile
        if len(self.parent_audio.shape) > 1:  # Stereo
            tactile_audio = np.zeros((len(self.parent_audio), self.parent_audio.shape[1]), dtype=self.parent_audio.dtype)
        else:  # Mono
            tactile_audio = np.zeros(len(self.parent_audio), dtype=self.parent_audio.dtype)
        
        # Track the injections for verification
        injection_log = {
            'looming': [],
            'tactile': []
        }
        
        # Inject looming stimuli
        for inject in injection_map['looming']:
            time_ms = inject['time_ms']
            stimulus_type = inject['stimulus_type']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            retention_ms = inject.get('retention_ms')  # New field added in validation
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            stimulus_data = self.looming_stimuli.get(stimulus_type)
            if stimulus_data is None:
                logging.warning(f"No stimulus data for type '{stimulus_type}'")
                continue
            
            # Check if injection fits within audio bounds
            if start_sample + len(stimulus_data) <= len(looming_audio):
                # Handle stereo/mono differences
                if len(looming_audio.shape) > 1 and len(stimulus_data.shape) > 1:
                    # Both are stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                elif len(looming_audio.shape) > 1 and len(stimulus_data.shape) == 1:
                    # Looming is stereo, stimulus is mono
                    for ch in range(looming_audio.shape[1]):
                        looming_audio[start_sample:start_sample+len(stimulus_data), ch] += stimulus_data
                elif len(looming_audio.shape) == 1 and len(stimulus_data.shape) > 1:
                    # Looming is mono, stimulus is stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += np.mean(stimulus_data, axis=1)
                else:
                    # Both are mono
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                
                # Log successful injection
                log_entry = {
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'stimulus_type': stimulus_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample,
                    'timestamp': inject.get('timestamp', 'N/A')
                }
                
                # Add retention timestamp if available (for validation)
                if retention_ms is not None:
                    log_entry['retention_ms'] = retention_ms
                
                injection_log['looming'].append(log_entry)
                
                if self.config['debug_mode'] and len(injection_log['looming']) <= 5:
                    # Show detailed timing for first few injections
                    logging.info(f"Injected looming ({stimulus_type}) at {time_ms:.1f} ms (trial {trial_number})")
                    if retention_ms is not None:
                        delay_after_retention = time_ms - retention_ms
                        logging.info(f"  - Delay after retention: {delay_after_retention:.1f} ms")
            else:
                logging.warning(f"Looming stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        # Inject tactile stimuli
        for inject in injection_map['tactile']:
            time_ms = inject['time_ms']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            looming_ms = inject.get('looming_ms')  # New field added in validation
            retention_ms = inject.get('retention_ms')  # New field added in validation
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Check if injection fits within audio bounds
            if start_sample + len(self.tactile_stimulus) <= len(tactile_audio):
                # Handle stereo/mono differences
                if len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) > 1:
                    # Both are stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                elif len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) == 1:
                    # Tactile audio is stereo, stimulus is mono
                    for ch in range(tactile_audio.shape[1]):
                        tactile_audio[start_sample:start_sample+len(self.tactile_stimulus), ch] += self.tactile_stimulus
                elif len(tactile_audio.shape) == 1 and len(self.tactile_stimulus.shape) > 1:
                    # Tactile audio is mono, stimulus is stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += np.mean(self.tactile_stimulus, axis=1)
                else:
                    # Both are mono
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                
                # Log successful injection
                log_entry = {
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample,
                    'timestamp': inject.get('timestamp', 'N/A')
                }
                
                # Add looming and retention timestamps if available (for validation)
                if looming_ms is not None:
                    log_entry['looming_ms'] = looming_ms
                if retention_ms is not None:
                    log_entry['retention_ms'] = retention_ms
                
                injection_log['tactile'].append(log_entry)
                
                if self.config['debug_mode'] and len(injection_log['tactile']) <= 5:
                    # Show detailed timing for first few injections
                    logging.info(f"Injected tactile at {time_ms:.1f} ms (trial {trial_number})")
                    if looming_ms is not None:
                        soa_ms = time_ms - looming_ms
                        logging.info(f"  - SOA (after looming): {soa_ms:.1f} ms")
                    if retention_ms is not None and trial_type == 'baseline':
                        delay_after_retention = time_ms - retention_ms
                        logging.info(f"  - Delay after retention: {delay_after_retention:.1f} ms")
            else:
                logging.warning(f"Tactile stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        if self.config['debug_mode']:
            logging.info(f"Injected stimuli:")
            logging.info(f"  Looming injections completed: {len(injection_log['looming'])}")
            logging.info(f"  Tactile injections completed: {len(injection_log['tactile'])}")
        
        return looming_audio, tactile_audio, injection_log
    
    def save_audio_files(self, participant_id, looming_audio, tactile_audio, injection_log, validation_issues=None):
        """
        Save the generated audio files and validation report.
        
        Args:
            participant_id: Participant ID
            looming_audio: Looming audio data
            tactile_audio: Tactile audio data
            injection_log: Dictionary with injection details
            validation_issues: List of validation issues found
            
        Returns:
            Tuple of file paths (looming_path, tactile_path, validation_report_path)
        """
        # Create filenames
        looming_filename = f"participant_{participant_id}_design_looming.wav"
        tactile_filename = f"participant_{participant_id}_design_tactile.wav"
        validation_filename = f"participant_{participant_id}_validation_report.txt"
        
        # Create full paths
        looming_path = os.path.join(self.output_dir, looming_filename)
        tactile_path = os.path.join(self.output_dir, tactile_filename)
        validation_path = os.path.join(self.output_dir, validation_filename)
        
        # Save audio files
        sf.write(looming_path, looming_audio, self.sample_rate)
        sf.write(tactile_path, tactile_audio, self.sample_rate)
        
        # Create validation report instead of injection log CSV
        with open(validation_path, 'w') as f:
            f.write(f"# Validation Report for Participant {participant_id}\n")
            f.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write("## Timestamp Hierarchy Validation\n")
            f.write("Required order: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp\n\n")
            
            if validation_issues and len(validation_issues) > 0:
                f.write(f"VALIDATION FAILED: Found {len(validation_issues)} timestamp ordering issues\n")
                f.write("The following trials have invalid timestamp ordering:\n\n")
                
                for issue in validation_issues:
                    f.write(f"Trial {issue['trial_number']}: {issue['description']}\n")
                    # Include the actual timestamps for reference
                    for ts_key in [k for k in issue if k.endswith('_ts')]:
                        f.write(f"  {ts_key}: {issue[ts_key]}\n")
                    f.write("\n")
                
                f.write("These invalid trials were SKIPPED in the audio generation.\n\n")
            else:
                f.write("VALIDATION PASSED: All timestamps follow the required ordering.\n\n")
            
            # Summarize successful injection stats
            f.write("## Audio Generation Summary\n")
            f.write(f"Successfully injected looming stimuli: {len(injection_log['looming'])}\n")
            f.write(f"Successfully injected tactile stimuli: {len(injection_log['tactile'])}\n\n")
            
            # List all injected stimuli for reference
            f.write("## Successful Injections\n")
            f.write("### Looming Stimuli\n")
            for i, inj in enumerate(injection_log['looming']):
                f.write(f"{i+1}. Trial {inj['trial_number']} ({inj['trial_type']}): ")
                f.write(f"{inj['stimulus_type']} at {inj.get('timestamp', 'N/A')} ")
                f.write(f"({inj['time_ms']:.1f} ms)\n")
            
            f.write("\n### Tactile Stimuli\n")
            for i, inj in enumerate(injection_log['tactile']):
                f.write(f"{i+1}. Trial {inj['trial_number']} ({inj['trial_type']}): ")
                f.write(f"at {inj.get('timestamp', 'N/A')} ")
                f.write(f"({inj['time_ms']:.1f} ms)\n")
            
            # Verify SOA values where applicable
            f.write("\n## SOA Verification\n")
            soa_verification = []
            looming_dict = {inj['trial_number']: inj for inj in injection_log['looming']}
            tactile_dict = {inj['trial_number']: inj for inj in injection_log['tactile']}
            
            for trial_num, looming in looming_dict.items():
                if trial_num in tactile_dict:
                    tactile = tactile_dict[trial_num]
                    soa_ms = tactile['time_ms'] - looming['time_ms']
                    
                    f.write(f"Trial {trial_num}: SOA = {soa_ms:.1f} ms ")
                    f.write(f"(Looming: {looming.get('timestamp', 'N/A')}, ")
                    f.write(f"Tactile: {tactile.get('timestamp', 'N/A')})\n")
        
        if self.config['debug_mode']:
            logging.info(f"Saved audio files and validation report for participant {participant_id}:")
            logging.info(f"  Looming: {looming_path}")
            logging.info(f"  Tactile: {tactile_path}")
            logging.info(f"  Validation Report: {validation_path}")
            
            # Log validation summary
            if validation_issues and len(validation_issues) > 0:
                logging.warning(f"  ⚠️ VALIDATION FAILED: {len(validation_issues)} invalid timestamp orderings")
            else:
                logging.info(f"  ✓ VALIDATION PASSED: All timestamps follow the required ordering")
        
        return looming_path, tactile_path, validation_path
    
    def verify_audio_sync(self, injection_log):
        """
        Verify audio synchronization by comparing expected SOA values against actual.
        
        Args:
            injection_log: Dictionary with injection details
            
        Returns:
            DataFrame with verification results
        """
        verification_results = []
        
        # Create dict for faster lookup
        looming_dict = {inject['trial_number']: inject for inject in injection_log['looming']}
        tactile_dict = {inject['trial_number']: inject for inject in injection_log['tactile']}
        
        # Check each looming trial that should have a tactile counterpart
        for trial_num, looming in looming_dict.items():
            if looming['trial_type'] == 'catch':
                continue  # Skip catch trials
                
            if trial_num in tactile_dict:
                tactile = tactile_dict[trial_num]
                
                actual_soa_ms = tactile['time_ms'] - looming['time_ms']
                
                verification_results.append({
                    'trial_number': trial_num,
                    'trial_type': looming['trial_type'],
                    'looming_time_ms': looming['time_ms'],
                    'tactile_time_ms': tactile['time_ms'],
                    'actual_soa_ms': actual_soa_ms
                })
        
        results_df = pd.DataFrame(verification_results)
        
        if self.config['debug_mode'] and not results_df.empty:
            # Print first few verification results
            logging.info(f"Verification sample (first 5 trials):")
            
            for _, row in results_df.head(5).iterrows():
                logging.info(f"  Trial {row['trial_number']} ({row['trial_type']}): " 
                          f"SOA = {row['actual_soa_ms']:.2f}ms")
            
            # Calculate statistics
            mean_soa = results_df['actual_soa_ms'].mean()
            min_soa = results_df['actual_soa_ms'].min()
            max_soa = results_df['actual_soa_ms'].max()
            
            logging.info(f"SOA statistics:")
            logging.info(f"  Mean: {mean_soa:.2f}ms")
            logging.info(f"  Min: {min_soa:.2f}ms")
            logging.info(f"  Max: {max_soa:.2f}ms")
        
        return results_df
    
    def process_participant(self, participant_id):
        """
        Process a single participant.
        
        Args:
            participant_id: Participant ID
            
        Returns:
            Tuple of (success, file_paths)
        """
        try:
            if self.config['debug_mode']:
                logging.info(f"\n=== Processing participant {participant_id} ===")
            
            # Load design CSV
            design_df = self.load_participant_design(participant_id)
            
            # Create injection map with timestamp validation
            injection_map, validation_issues = self.create_injection_map(design_df)
            
            # Log validation summary
            if validation_issues:
                logging.warning(f"Found {len(validation_issues)} timestamp hierarchy issues")
                logging.warning("Hierarchy should be: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp")
                for issue in validation_issues[:5]:  # Show first 5 issues
                    logging.warning(f"  Trial {issue['trial_number']}: {issue['description']}")
                if len(validation_issues) > 5:
                    logging.warning(f"  ... and {len(validation_issues) - 5} more issues")
            
            # Create participant audio
            looming_audio, tactile_audio, injection_log = self.create_participant_audio(
                participant_id, injection_map)
            
            # Verify synchronization
            verification = self.verify_audio_sync(injection_log)
            
            # Save audio files and validation report
            looming_path, tactile_path, log_path = self.save_audio_files(
                participant_id, looming_audio, tactile_audio, injection_log, validation_issues)
            
            return True, (looming_path, tactile_path, log_path)
            
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """
        Run audio generation for multiple participants.
        
        Args:
            participant_ids: List of participant IDs or None for all
            
        Returns:
            Dictionary with results
        """
        # If no participant IDs provided, find all design CSVs
        if participant_ids is None:
            # Find all participant design files
            design_dir = self.config['paths']['design_dir']
            design_files = [f for f in os.listdir(design_dir) if f.startswith('participant_') and f.endswith('_design.csv')]
            
            # Extract participant IDs
            participant_ids = []
            for filename in design_files:
                match = re.match(r'participant_(\d+)_design\.csv', filename)
                if match:
                    participant_ids.append(int(match.group(1)))
            
            participant_ids.sort()  # Sort numerically
        
        if self.config['debug_mode']:
            logging.info(f"=== Starting PPSAudioGenerator ===")
            logging.info(f"Processing {len(participant_ids)} participants: {participant_ids}")
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        # Summary
        if self.config['debug_mode']:
            success_count = sum(1 for info in results.values() if info['success'])
            logging.info(f"\n=== GENERATION SUMMARY ===")
            logging.info(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        
        return results

def main():
    """Main function to run the audio generator."""
    print("\n===== PPS EXPERIMENT AUDIO GENERATOR =====")
    print("This script generates participant-specific audio files with precise timing")
    print("It validates timestamp ordering: retentionslot < looming < tactile\n")
    
    # Configuration
    base_config = CONFIG.copy()
    
    # Auto-detect parent audio file (already implemented in the generator class)
    
    # By default, process all participant designs found in the directory
    participant_ids = None
    print("\nProcessing all available participants")
    
    # Create generator
    try:
        generator = PPSAudioGenerator(base_config)
        
        print(f"\nConfiguration:")
        print(f"  Parent audio: {generator.parent_audio_path}")
        print(f"  Design directory: {base_config['paths']['design_dir']}")
        print(f"  Output directory: {base_config['paths']['output_dir']}")
        print(f"  Timestamp hierarchy: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp")
        
        # Run generator
        results = generator.run(participant_ids)
        
        # Print summary
        print("\n===== GENERATION SUMMARY =====")
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        print("\nVALIDATION INFORMATION:")
        print("  - Only stimuli with correct timestamp ordering were included")
        print("  - Detailed reports saved as participant_*_validation_report.txt")
        print("  - Timestamps with validation issues were SKIPPED")
        
        for pid, info in results.items():
            if info['success']:
                looming_path, tactile_path, validation_path = info['paths']
                print(f"\nParticipant {pid}: Files generated")
                print(f"  Looming: {os.path.basename(looming_path)}")
                print(f"  Tactile: {os.path.basename(tactile_path)}")
                print(f"  Validation: {os.path.basename(validation_path)}")
            else:
                print(f"\nParticipant {pid}: ERROR - audio generation failed.")
                
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()
    
    print("\nAudio generation complete.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT AUDIO GENERATOR =====
This script generates participant-specific audio files with precise timing
It validates timestamp ordering: retentionslot < looming < tactile


Processing all available participants


2025-03-27 21:19:32,174 - INFO - Loaded looming stimulus 'blue' from front_az0_FABIAN_HRIR_blue.wav
2025-03-27 21:19:32,178 - INFO - Loaded looming stimulus 'brown' from front_az0_FABIAN_HRIR_brown.wav
2025-03-27 21:19:32,181 - INFO - Loaded looming stimulus 'pink' from front_az0_FABIAN_HRIR_pink.wav
2025-03-27 21:19:32,186 - INFO - Loaded looming stimulus 'white' from front_az0_FABIAN_HRIR_white.wav
2025-03-27 21:19:32,188 - INFO - Loaded parent audio: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete_x5_20250327_202030.wav
2025-03-27 21:19:32,189 - INFO -   Duration: 1775.66 seconds
2025-03-27 21:19:32,190 - INFO -   Sample Rate: 48000 Hz
2025-03-27 21:19:32,192 - INFO -   Shape: (85231648, 2)
2025-03-27 21:19:32,193 - INFO -   Looming 'blue': 2.76 seconds, (132300, 2)
2025-03-27 21:19:32,193 - INFO -   Looming 'brown': 2.76 seconds, (132300, 2)
2025-03-27 21:19:32,193 - INFO -   Looming 'pink': 2.76 seconds, (132300, 2


Configuration:
  Parent audio: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete_x5_20250327_202030.wav
  Design directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
  Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio
  Timestamp hierarchy: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp


2025-03-27 21:19:32,781 - INFO - Injected looming (white) at 104600.0 ms (trial 10)
2025-03-27 21:19:32,782 - INFO -   - Delay after retention: 500.0 ms
2025-03-27 21:19:32,783 - INFO - Injected looming (pink) at 112600.0 ms (trial 11)
2025-03-27 21:19:32,783 - INFO -   - Delay after retention: 500.0 ms
2025-03-27 21:19:32,785 - INFO - Injected looming (brown) at 120400.0 ms (trial 12)
2025-03-27 21:19:32,786 - INFO -   - Delay after retention: 300.0 ms
2025-03-27 21:19:32,789 - INFO - Injected looming (blue) at 136300.0 ms (trial 14)
2025-03-27 21:19:32,789 - INFO -   - Delay after retention: 200.0 ms
2025-03-27 21:19:32,790 - INFO - Injected looming (blue) at 144300.0 ms (trial 15)
2025-03-27 21:19:32,791 - INFO -   - Delay after retention: 100.0 ms
2025-03-27 21:19:32,839 - INFO - Injected tactile at 96700.0 ms (trial 9)
2025-03-27 21:19:32,839 - INFO -   - Delay after retention: 600.0 ms
2025-03-27 21:19:32,843 - INFO - Injected tactile at 106100.0 ms (trial 10)
2025-03-27 21:19:32


===== GENERATION SUMMARY =====
Successfully generated audio for 100/100 participants.

VALIDATION INFORMATION:
  - Only stimuli with correct timestamp ordering were included
  - Detailed reports saved as participant_*_validation_report.txt
  - Timestamps with validation issues were SKIPPED

Participant 0: Files generated
  Looming: participant_0_design_looming.wav
  Tactile: participant_0_design_tactile.wav
  Validation: participant_0_validation_report.txt

Participant 1: Files generated
  Looming: participant_1_design_looming.wav
  Tactile: participant_1_design_tactile.wav
  Validation: participant_1_validation_report.txt

Participant 2: Files generated
  Looming: participant_2_design_looming.wav
  Tactile: participant_2_design_tactile.wav
  Validation: participant_2_validation_report.txt

Participant 3: Files generated
  Looming: participant_3_design_looming.wav
  Tactile: participant_3_design_tactile.wav
  Validation: participant_3_validation_report.txt

Participant 4: Files genera

In [10]:
def validate_design_vs_wav_markers(self, design_df):
        """
        Validate design CSV timestamps against actual WAV file markers.
        
        Args:
            design_df: DataFrame with design data
            
        Returns:
            Tuple of (validation_passed, validation_issues, corrected_design_df)
        """
        if not self.wav_markers:
            logging.warning("No WAV markers available for validation - skipping validation")
            return True, [], design_df
        
        # Create a copy of the design DataFrame that we can modify
        corrected_df = design_df.copy()
        
        # Track validation issues
        validation_issues = []
        
        # Check if retentionslot_timestamp column exists
        if 'retentionslot_timestamp' not in design_df.columns:
            logging.warning("No retentionslot_timestamp column in design CSV - cannot validate against WAV markers")
            return False, [{
                'issue_type': 'missing_column',
                'description': "No retentionslot_timestamp column in design CSV"
            }], design_df
        
        # Convert WAV markers to milliseconds for easier comparison
        wav_marker_ms = {m['timestamp']: m['time_ms'] for m in self.wav_markers}
        wav_marker_phases = {m['timestamp']: m['breath_phase'] for m in self.wav_markers}
        
        # Convert all retention timestamps to milliseconds
        corrected_df['retention_ms'] = corrected_df['retentionslot_timestamp'].apply(parse_timestamp)
        
        # Add columns to track marker validation
        corrected_df['retention_in_wav'] = False
        corrected_df['retention_phase_matches'] = False
        
        # Count how many retention timestamps match WAV markers
        retention_matches = 0
        retention_mismatches = 0
        
        for idx, row in corrected_df.iterrows():
            if pd.isna(row['retentionslot_timestamp']):
                continue
                
            # Check if this retention timestamp exists in WAV markers
            if row['retentionslot_timestamp'] in wav_marker_ms:
                corrected_df.loc[idx, 'retention_in_wav'] = True
                retention_matches += 1
                
                # Check if breath phase matches
                csv_phase = row['breathphase'].lower() if 'breathphase' in row and not pd.isna(row['breathphase']) else None
                wav_phase = wav_marker_phases.get(row['retentionslot_timestamp'])
                
                if csv_phase and wav_phase and csv_phase == wav_phase:
                    corrected_df.loc[idx, 'retention_phase_matches'] = True
                else:
                    retention_mismatches += 1
                    # Track phase mismatch issue
                    validation_issues.append({
                        'trial_number': row.get('trial_number', idx),
                        'issue_type': 'phase_mismatch',
                        'description': f"Breath phase mismatch: CSV={csv_phase}, WAV={wav_phase}",
                        'retention_timestamp': row['retentionslot_timestamp']
                    })
            else:
                # Track missing marker issue
                validation_issues.append({
                    'trial_number': row.get('trial_number', idx),
                    'issue_type': 'missing_marker',
                    'description': f"Retention timestamp {row['retentionslot_timestamp']} not found in WAV markers",
                    'retention_timestamp': row['retentionslot_timestamp']
                })
        
        # Calculate validation results
        total_retention_timestamps = corrected_df['retentionslot_timestamp'].notna().sum()
        retention_match_percent = (retention_matches / total_retention_timestamps * 100) if total_retention_timestamps > 0 else 0
        
        logging.info(f"WAV marker validation results:")
        logging.info(f"  Total retention timestamps in CSV: {total_retention_timestamps}")
        logging.info(f"  Matching WAV markers: {retention_matches} ({retention_match_percent:.1f}%)")
        logging.info(f"  Phase mismatches: {retention_mismatches}")
        
        # Determine if validation passed based on threshold
        validation_threshold = 0.8  # 80% match required
        validation_passed = retention_match_percent >= validation_threshold * 100
        
        if validation_passed:
            logging.info(f"WAV marker validation PASSED: {retention_match_percent:.1f}% timestamps match")
        else:
            logging.warning(f"WAV marker validation FAILED: Only {retention_match_percent:.1f}% timestamps match (threshold: {validation_threshold*100:.0f}%)")
        
        return validation_passed, validation_issues, corrected_df    def extract_wav_markers(self, wav_file_path):
        """
        Extract markers directly from the WAV file.
        
        Args:
            wav_file_path: Path to the WAV file
            
        Returns:
            List of marker dictionaries with timing and phase information
        """
        markers = []
        
        try:
            # Get sample rate
            with wave.open(wav_file_path, 'rb') as wav:
                sample_rate = wav.getframerate()
            
            # Open WAV file in binary mode
            with open(wav_file_path, 'rb') as f:
                # Check RIFF header
                if f.read(4) != b'RIFF':
                    raise ValueError("Not a valid RIFF file")
                
                # Skip file size
                f.read(4)
                
                # Check WAVE format
                if f.read(4) != b'WAVE':
                    raise ValueError("Not a valid WAVE file")
                
                # Find the cue chunk
                chunk_id = None
                while True:
                    chunk_pos = f.tell()
                    chunk_id = f.read(4)
                    
                    if not chunk_id or len(chunk_id) < 4:
                        break  # End of file or error
                    
                    chunk_size = int.from_bytes(f.read(4), byteorder='little')
                    
                    if chunk_id == b'cue ':
                        # Found cue chunk
                        num_cues = int.from_bytes(f.read(4), byteorder='little')
                        logging.info(f"Found {num_cues} cue points in WAV file")
                        
                        # Read each cue point
                        for i in range(num_cues):
                            # Read cue point ID
                            cue_id = int.from_bytes(f.read(4), byteorder='little')
                            
                            # Read position
                            position = int.from_bytes(f.read(4), byteorder='little')
                            
                            # Skip remaining cue point data (16 bytes)
                            f.read(16)
                            
                            # Convert position to time in milliseconds
                            time_ms = (position / sample_rate) * 1000
                            timestamp = convert_ms_to_timestamp(time_ms)
                            
                            # Determine breath phase from cue ID
                            # By convention: odd IDs (1,3,5...) are inhale, even IDs (2,4,6...) are exhale
                            breath_phase = 'inhale' if cue_id % 2 == 1 else 'exhale'
                            
                            markers.append({
                                'cue_id': cue_id,
                                'position': position,
                                'time_ms': time_ms,
                                'timestamp': timestamp,
                                'breath_phase': breath_phase
                            })
                        
                        break  # Found and processed the cue chunk
                    else:
                        # Skip this chunk
                        f.seek(chunk_pos + 8 + chunk_size)
            
            # Sort markers by position
            markers.sort(key=lambda x: x['position'])
            
            if markers:
                logging.info(f"Extracted {len(markers)} markers from WAV file")
                logging.info(f"First marker: {markers[0]['timestamp']} ({markers[0]['breath_phase']})")
                logging.info(f"Last marker: {markers[-1]['timestamp']} ({markers[-1]['breath_phase']})")
                
                # Log a sample of markers
                for i in range(min(5, len(markers))):
                    logging.info(f"  Marker {i+1}: {markers[i]['timestamp']} ({markers[i]['breath_phase']})")
            else:
                logging.warning("No markers found in WAV file!")
                
        except Exception as e:
            logging.error(f"Error extracting markers from WAV file: {e}")
            traceback.print_exc()
        
        return markers#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Audio Generator

This script generates participant-specific audio files for a Peripersonal Space (PPS) experiment.
It reads design CSV files from the ExperimentLog directory and creates:
1. A looming audio file with the parent audio and looming stimuli overlaid at specified timestamps
2. A tactile audio file with tactile stimuli at specified timestamps

Author: Claude
"""

import os
import numpy as np
import pandas as pd
import soundfile as sf
import re
import logging
from pathlib import Path
import traceback

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration
CONFIG = {
    'paths': {
        # Find the most recent parent audio file with "BoxBreathing_complete" prefix
        'design_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles",
    },
    'debug_mode': True,
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav'
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# Helper function for parsing timestamps
def parse_timestamp(timestamp):
    """
    Parse timestamp in format MM:SS.S to milliseconds.
    
    Args:
        timestamp: String in format "MM:SS.S"
        
    Returns:
        Float: milliseconds
    """
    if pd.isna(timestamp):
        return None
        
    match = re.match(r'(\d+):(\d+\.\d+)', timestamp)
    if match:
        minutes, seconds = match.groups()
        return (int(minutes) * 60 + float(seconds)) * 1000
    return None

class PPSAudioGenerator:
    """Generates participant-specific audio files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.output_dir = self.config['paths']['output_dir']
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Find parent audio file (most recent one with BoxBreathing_complete prefix)
        self.parent_audio_path = self._find_parent_audio()
        if not self.parent_audio_path:
            raise FileNotFoundError("Could not find parent audio file with BoxBreathing_complete prefix")
        
        # Load parent audio
        self.parent_audio, self.sample_rate = self._load_audio(self.parent_audio_path)
        
        # Extract markers directly from the parent WAV file
        self.wav_markers = self.extract_wav_markers(self.parent_audio_path)
        
        # Load stimulus files
        self.looming_stimuli = self._load_looming_stimuli()
        self.tactile_stimulus, _ = self._load_audio(os.path.join(
            self.config['paths']['stimulus_dir'], TACTILE_STIMULUS))
        
        if self.config['debug_mode']:
            self._log_audio_info()
    
    def _find_parent_audio(self):
        """Find the most recent parent audio file in the output directory."""
        audio_dir = self.config['paths']['output_dir']
        parent_files = [f for f in os.listdir(audio_dir) 
                      if f.startswith("BoxBreathing_complete") and f.endswith(".wav")]
        
        if not parent_files:
            return None
        
        # Sort by creation time (most recent first)
        parent_files.sort(key=lambda f: os.path.getmtime(os.path.join(audio_dir, f)), reverse=True)
        
        return os.path.join(audio_dir, parent_files[0])
    
    def _load_audio(self, file_path):
        """Load audio file and return data and sample rate."""
        try:
            data, sample_rate = sf.read(file_path)
            return data, sample_rate
        except Exception as e:
            logging.error(f"Error loading audio file {file_path}: {e}")
            raise
    
    def _load_looming_stimuli(self):
        """Load all looming stimulus files."""
        looming_stimuli = {}
        for key, filename in LOOMING_STIMULI.items():
            file_path = os.path.join(self.config['paths']['stimulus_dir'], filename)
            try:
                data, _ = self._load_audio(file_path)
                looming_stimuli[key] = data
                if self.config['debug_mode']:
                    logging.info(f"Loaded looming stimulus '{key}' from {filename}")
            except Exception as e:
                logging.warning(f"Could not load looming stimulus '{key}' from {filename}: {e}")
        return looming_stimuli
    
    def _log_audio_info(self):
        """Log information about loaded audio files for debugging."""
        logging.info(f"Loaded parent audio: {self.parent_audio_path}")
        logging.info(f"  Duration: {len(self.parent_audio)/self.sample_rate:.2f} seconds")
        logging.info(f"  Sample Rate: {self.sample_rate} Hz")
        logging.info(f"  Shape: {self.parent_audio.shape}")
        logging.info(f"  WAV markers extracted: {len(self.wav_markers)}")
        
        for key, data in self.looming_stimuli.items():
            logging.info(f"  Looming '{key}': {len(data)/self.sample_rate:.2f} seconds, {data.shape}")
        
        logging.info(f"  Tactile: {len(self.tactile_stimulus)/self.sample_rate:.2f} seconds, {self.tactile_stimulus.shape}")
        
        logging.info("=== VALIDATION STRATEGY ===")
        logging.info("1. Validating design CSV timestamps against actual WAV file markers")
        logging.info("2. Ensuring timestamp hierarchy: retention phase < looming < tactile")
        logging.info("3. Skipping any trials with invalid timing")
    
    def load_participant_design(self, participant_id):
        """
        Load participant design CSV and parse timestamps.
        
        Args:
            participant_id: Participant ID number
            
        Returns:
            DataFrame containing the trial design
        """
        design_path = os.path.join(
            self.config['paths']['design_dir'], 
            f"participant_{participant_id}_design.csv")
        
        try:
            design_df = pd.read_csv(design_path)
            
            # Parse timestamp columns to milliseconds
            for col in ['looming_stimulus_timestamp', 'tactile_stimulus_timestamp']:
                if col in design_df.columns:
                    design_df[f'{col}_ms'] = design_df[col].apply(parse_timestamp)
            
            if self.config['debug_mode']:
                max_trial = design_df['trial_number'].max() if 'trial_number' in design_df.columns else len(design_df)
                logging.info(f"Loaded design for participant {participant_id}")
                logging.info(f"  Total trials: {len(design_df)}")
                logging.info(f"  Max trial number: {max_trial}")
                if 'trial_type' in design_df.columns:
                    logging.info(f"  Trial types: {design_df['trial_type'].value_counts().to_dict()}")
            
            return design_df
            
        except Exception as e:
            logging.error(f"Error loading design for participant {participant_id}: {e}")
            raise
    
    def create_injection_map(self, design_df):
        """
        Create a map of where to inject looming and tactile stimuli.
        Validates timestamp hierarchy: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp
        
        Args:
            design_df: DataFrame with the design
            
        Returns:
            Dictionary with injection details and validation results
        """
        # First validate design against WAV markers
        wav_validation_passed, wav_validation_issues, corrected_df = self.validate_design_vs_wav_markers(design_df)
        
        # Use the corrected design dataframe
        design_df = corrected_df
        
        injection_map = {
            'looming': [],
            'tactile': []
        }
        
        # Track additional validation issues (timestamp ordering)
        validation_issues = wav_validation_issues.copy()
        
        # Process each trial
        for _, row in design_df.iterrows():
            # Skip if no trial_type column or if it's a practice trial
            if 'trial_type' not in row or row['trial_type'] == 'practice':
                continue
            
            trial_type = row['trial_type']
            trial_number = row.get('trial_number', 0)
            
            # Get retention timestamp (should be earliest)
            retention_ms = None
            retention_in_wav = False
            if 'retention_ms' in row and not pd.isna(row['retention_ms']):
                retention_ms = row['retention_ms']
                retention_in_wav = row.get('retention_in_wav', False)
                
            # Get looming timestamp
            looming_ms = None
            if trial_type in ['inhalation', 'exhalation', 'catch'] and 'looming_stimulus_timestamp' in row and not pd.isna(row['looming_stimulus_timestamp']):
                looming_ms = parse_timestamp(row['looming_stimulus_timestamp'])
                
                # Validate looming comes after retention
                if retention_ms is not None and looming_ms is not None and looming_ms <= retention_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Looming timestamp ({looming_ms} ms) should be after retention timestamp ({retention_ms} ms)",
                        'retention_ts': row.get('retentionslot_timestamp', 'N/A'),
                        'looming_ts': row.get('looming_stimulus_timestamp', 'N/A'),
                        'retention_in_wav': retention_in_wav
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                    
                if looming_ms is not None:
                    stim_type = row['stimulus_type'] if 'stimulus_type' in row and not pd.isna(row['stimulus_type']) else 'white'
                    
                    if stim_type in self.looming_stimuli:
                        injection_map['looming'].append({
                            'time_ms': looming_ms,
                            'timestamp': row['looming_stimulus_timestamp'],
                            'stimulus_type': stim_type,
                            'trial_number': trial_number,
                            'trial_type': trial_type,
                            'retention_ms': retention_ms,
                            'retention_in_wav': retention_in_wav
                        })
            
            # Get tactile timestamp
            tactile_ms = None
            if trial_type in ['inhalation', 'exhalation', 'baseline'] and 'tactile_stimulus_timestamp' in row and not pd.isna(row['tactile_stimulus_timestamp']):
                tactile_ms = parse_timestamp(row['tactile_stimulus_timestamp'])
                
                # Validate tactile comes after looming (if applicable)
                if looming_ms is not None and tactile_ms is not None and tactile_ms <= looming_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Tactile timestamp ({tactile_ms} ms) should be after looming timestamp ({looming_ms} ms)",
                        'looming_ts': row.get('looming_stimulus_timestamp', 'N/A'),
                        'tactile_ts': row.get('tactile_stimulus_timestamp', 'N/A')
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                
                # For baseline trials, validate tactile comes after retention
                if trial_type == 'baseline' and retention_ms is not None and tactile_ms is not None and tactile_ms <= retention_ms:
                    issue = {
                        'trial_number': trial_number,
                        'issue_type': 'timestamp_ordering',
                        'description': f"Baseline tactile timestamp ({tactile_ms} ms) should be after retention timestamp ({retention_ms} ms)",
                        'retention_ts': row.get('retentionslot_timestamp', 'N/A'),
                        'tactile_ts': row.get('tactile_stimulus_timestamp', 'N/A'),
                        'retention_in_wav': retention_in_wav
                    }
                    validation_issues.append(issue)
                    logging.warning(f"Trial {trial_number}: {issue['description']}")
                    # Skip this stimulus due to invalid timing
                    continue
                    
                if tactile_ms is not None:
                    injection_map['tactile'].append({
                        'time_ms': tactile_ms,
                        'timestamp': row['tactile_stimulus_timestamp'],
                        'trial_number': trial_number,
                        'trial_type': trial_type,
                        'looming_ms': looming_ms,
                        'retention_ms': retention_ms,
                        'retention_in_wav': retention_in_wav
                    })
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
            if validation_issues:
                logging.warning(f"  Found {len(validation_issues)} validation issues!")
            else:
                logging.info("  All timestamps validated successfully")
                
            # Report WAV marker validation results
            if not wav_validation_passed:
                logging.warning("  WAV marker validation failed - proceed with caution!")
        
        return injection_map, validation_issues
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
        
        return injection_map
    
    def create_participant_audio(self, participant_id, injection_map):
        """
        Create participant-specific audio by overlaying stimuli on parent audio.
        
        Args:
            participant_id: Participant ID
            injection_map: Dictionary with injection details
            
        Returns:
            Tuple of (looming_audio, tactile_audio, injection_log)
        """
        # Create copies of parent audio for looming
        looming_audio = self.parent_audio.copy()
        
        # Create empty audio of same length for tactile
        if len(self.parent_audio.shape) > 1:  # Stereo
            tactile_audio = np.zeros((len(self.parent_audio), self.parent_audio.shape[1]), dtype=self.parent_audio.dtype)
        else:  # Mono
            tactile_audio = np.zeros(len(self.parent_audio), dtype=self.parent_audio.dtype)
        
        # Track the injections for verification
        injection_log = {
            'looming': [],
            'tactile': []
        }
        
        # Inject looming stimuli
        for inject in injection_map['looming']:
            time_ms = inject['time_ms']
            stimulus_type = inject['stimulus_type']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            retention_ms = inject.get('retention_ms')  # New field added in validation
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            stimulus_data = self.looming_stimuli.get(stimulus_type)
            if stimulus_data is None:
                logging.warning(f"No stimulus data for type '{stimulus_type}'")
                continue
            
            # Check if injection fits within audio bounds
            if start_sample + len(stimulus_data) <= len(looming_audio):
                # Handle stereo/mono differences
                if len(looming_audio.shape) > 1 and len(stimulus_data.shape) > 1:
                    # Both are stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                elif len(looming_audio.shape) > 1 and len(stimulus_data.shape) == 1:
                    # Looming is stereo, stimulus is mono
                    for ch in range(looming_audio.shape[1]):
                        looming_audio[start_sample:start_sample+len(stimulus_data), ch] += stimulus_data
                elif len(looming_audio.shape) == 1 and len(stimulus_data.shape) > 1:
                    # Looming is mono, stimulus is stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += np.mean(stimulus_data, axis=1)
                else:
                    # Both are mono
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                
                # Log successful injection
                log_entry = {
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'stimulus_type': stimulus_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample,
                    'timestamp': inject.get('timestamp', 'N/A')
                }
                
                # Add retention timestamp if available (for validation)
                if retention_ms is not None:
                    log_entry['retention_ms'] = retention_ms
                
                injection_log['looming'].append(log_entry)
                
                if self.config['debug_mode'] and len(injection_log['looming']) <= 5:
                    # Show detailed timing for first few injections
                    logging.info(f"Injected looming ({stimulus_type}) at {time_ms:.1f} ms (trial {trial_number})")
                    if retention_ms is not None:
                        delay_after_retention = time_ms - retention_ms
                        logging.info(f"  - Delay after retention: {delay_after_retention:.1f} ms")
            else:
                logging.warning(f"Looming stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        # Inject tactile stimuli
        for inject in injection_map['tactile']:
            time_ms = inject['time_ms']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            looming_ms = inject.get('looming_ms')  # New field added in validation
            retention_ms = inject.get('retention_ms')  # New field added in validation
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Check if injection fits within audio bounds
            if start_sample + len(self.tactile_stimulus) <= len(tactile_audio):
                # Handle stereo/mono differences
                if len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) > 1:
                    # Both are stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                elif len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) == 1:
                    # Tactile audio is stereo, stimulus is mono
                    for ch in range(tactile_audio.shape[1]):
                        tactile_audio[start_sample:start_sample+len(self.tactile_stimulus), ch] += self.tactile_stimulus
                elif len(tactile_audio.shape) == 1 and len(self.tactile_stimulus.shape) > 1:
                    # Tactile audio is mono, stimulus is stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += np.mean(self.tactile_stimulus, axis=1)
                else:
                    # Both are mono
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                
                # Log successful injection
                log_entry = {
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample,
                    'timestamp': inject.get('timestamp', 'N/A')
                }
                
                # Add looming and retention timestamps if available (for validation)
                if looming_ms is not None:
                    log_entry['looming_ms'] = looming_ms
                if retention_ms is not None:
                    log_entry['retention_ms'] = retention_ms
                
                injection_log['tactile'].append(log_entry)
                
                if self.config['debug_mode'] and len(injection_log['tactile']) <= 5:
                    # Show detailed timing for first few injections
                    logging.info(f"Injected tactile at {time_ms:.1f} ms (trial {trial_number})")
                    if looming_ms is not None:
                        soa_ms = time_ms - looming_ms
                        logging.info(f"  - SOA (after looming): {soa_ms:.1f} ms")
                    if retention_ms is not None and trial_type == 'baseline':
                        delay_after_retention = time_ms - retention_ms
                        logging.info(f"  - Delay after retention: {delay_after_retention:.1f} ms")
            else:
                logging.warning(f"Tactile stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        if self.config['debug_mode']:
            logging.info(f"Injected stimuli:")
            logging.info(f"  Looming injections completed: {len(injection_log['looming'])}")
            logging.info(f"  Tactile injections completed: {len(injection_log['tactile'])}")
        
        return looming_audio, tactile_audio, injection_log
    
    def save_audio_files(self, participant_id, looming_audio, tactile_audio, injection_log, validation_issues=None):
        """
        Save the generated audio files and validation report.
        
        Args:
            participant_id: Participant ID
            looming_audio: Looming audio data
            tactile_audio: Tactile audio data
            injection_log: Dictionary with injection details
            validation_issues: List of validation issues found
            
        Returns:
            Tuple of file paths (looming_path, tactile_path, validation_report_path)
        """
        # Create filenames
        looming_filename = f"participant_{participant_id}_design_looming.wav"
        tactile_filename = f"participant_{participant_id}_design_tactile.wav"
        validation_filename = f"participant_{participant_id}_validation_report.txt"
        
        # Create full paths
        looming_path = os.path.join(self.output_dir, looming_filename)
        tactile_path = os.path.join(self.output_dir, tactile_filename)
        validation_path = os.path.join(self.output_dir, validation_filename)
        
        # Save audio files
        sf.write(looming_path, looming_audio, self.sample_rate)
        sf.write(tactile_path, tactile_audio, self.sample_rate)
        
        # Create validation report instead of injection log CSV
        with open(validation_path, 'w') as f:
            f.write(f"# Validation Report for Participant {participant_id}\n")
            f.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write("## Timestamp Hierarchy Validation\n")
            f.write("Required order: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp\n\n")
            
            if validation_issues and len(validation_issues) > 0:
                f.write(f"VALIDATION FAILED: Found {len(validation_issues)} timestamp ordering issues\n")
                f.write("The following trials have invalid timestamp ordering:\n\n")
                
                for issue in validation_issues:
                    f.write(f"Trial {issue['trial_number']}: {issue['description']}\n")
                    # Include the actual timestamps for reference
                    for ts_key in [k for k in issue if k.endswith('_ts')]:
                        f.write(f"  {ts_key}: {issue[ts_key]}\n")
                    f.write("\n")
                
                f.write("These invalid trials were SKIPPED in the audio generation.\n\n")
            else:
                f.write("VALIDATION PASSED: All timestamps follow the required ordering.\n\n")
            
            # Summarize successful injection stats
            f.write("## Audio Generation Summary\n")
            f.write(f"Successfully injected looming stimuli: {len(injection_log['looming'])}\n")
            f.write(f"Successfully injected tactile stimuli: {len(injection_log['tactile'])}\n\n")
            
            # List all injected stimuli for reference
            f.write("## Successful Injections\n")
            f.write("### Looming Stimuli\n")
            for i, inj in enumerate(injection_log['looming']):
                f.write(f"{i+1}. Trial {inj['trial_number']} ({inj['trial_type']}): ")
                f.write(f"{inj['stimulus_type']} at {inj.get('timestamp', 'N/A')} ")
                f.write(f"({inj['time_ms']:.1f} ms)\n")
            
            f.write("\n### Tactile Stimuli\n")
            for i, inj in enumerate(injection_log['tactile']):
                f.write(f"{i+1}. Trial {inj['trial_number']} ({inj['trial_type']}): ")
                f.write(f"at {inj.get('timestamp', 'N/A')} ")
                f.write(f"({inj['time_ms']:.1f} ms)\n")
            
            # Verify SOA values where applicable
            f.write("\n## SOA Verification\n")
            soa_verification = []
            looming_dict = {inj['trial_number']: inj for inj in injection_log['looming']}
            tactile_dict = {inj['trial_number']: inj for inj in injection_log['tactile']}
            
            for trial_num, looming in looming_dict.items():
                if trial_num in tactile_dict:
                    tactile = tactile_dict[trial_num]
                    soa_ms = tactile['time_ms'] - looming['time_ms']
                    
                    f.write(f"Trial {trial_num}: SOA = {soa_ms:.1f} ms ")
                    f.write(f"(Looming: {looming.get('timestamp', 'N/A')}, ")
                    f.write(f"Tactile: {tactile.get('timestamp', 'N/A')})\n")
        
        if self.config['debug_mode']:
            logging.info(f"Saved audio files and validation report for participant {participant_id}:")
            logging.info(f"  Looming: {looming_path}")
            logging.info(f"  Tactile: {tactile_path}")
            logging.info(f"  Validation Report: {validation_path}")
            
            # Log validation summary
            if validation_issues and len(validation_issues) > 0:
                logging.warning(f"  ⚠️ VALIDATION FAILED: {len(validation_issues)} invalid timestamp orderings")
            else:
                logging.info(f"  ✓ VALIDATION PASSED: All timestamps follow the required ordering")
        
        return looming_path, tactile_path, validation_path
    
    def verify_audio_sync(self, injection_log):
        """
        Verify audio synchronization by comparing expected SOA values against actual.
        
        Args:
            injection_log: Dictionary with injection details
            
        Returns:
            DataFrame with verification results
        """
        verification_results = []
        
        # Create dict for faster lookup
        looming_dict = {inject['trial_number']: inject for inject in injection_log['looming']}
        tactile_dict = {inject['trial_number']: inject for inject in injection_log['tactile']}
        
        # Check each looming trial that should have a tactile counterpart
        for trial_num, looming in looming_dict.items():
            if looming['trial_type'] == 'catch':
                continue  # Skip catch trials
                
            if trial_num in tactile_dict:
                tactile = tactile_dict[trial_num]
                
                actual_soa_ms = tactile['time_ms'] - looming['time_ms']
                
                verification_results.append({
                    'trial_number': trial_num,
                    'trial_type': looming['trial_type'],
                    'looming_time_ms': looming['time_ms'],
                    'tactile_time_ms': tactile['time_ms'],
                    'actual_soa_ms': actual_soa_ms
                })
        
        results_df = pd.DataFrame(verification_results)
        
        if self.config['debug_mode'] and not results_df.empty:
            # Print first few verification results
            logging.info(f"Verification sample (first 5 trials):")
            
            for _, row in results_df.head(5).iterrows():
                logging.info(f"  Trial {row['trial_number']} ({row['trial_type']}): " 
                          f"SOA = {row['actual_soa_ms']:.2f}ms")
            
            # Calculate statistics
            mean_soa = results_df['actual_soa_ms'].mean()
            min_soa = results_df['actual_soa_ms'].min()
            max_soa = results_df['actual_soa_ms'].max()
            
            logging.info(f"SOA statistics:")
            logging.info(f"  Mean: {mean_soa:.2f}ms")
            logging.info(f"  Min: {min_soa:.2f}ms")
            logging.info(f"  Max: {max_soa:.2f}ms")
        
        return results_df
    
    def process_participant(self, participant_id):
        """
        Process a single participant.
        
        Args:
            participant_id: Participant ID
            
        Returns:
            Tuple of (success, file_paths)
        """
        try:
            if self.config['debug_mode']:
                logging.info(f"\n=== Processing participant {participant_id} ===")
            
            # Load design CSV
            design_df = self.load_participant_design(participant_id)
            
            # Create injection map with timestamp validation
            injection_map, validation_issues = self.create_injection_map(design_df)
            
            # Log validation summary
            if validation_issues:
                logging.warning(f"Found {len(validation_issues)} timestamp hierarchy issues")
                logging.warning("Hierarchy should be: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp")
                for issue in validation_issues[:5]:  # Show first 5 issues
                    logging.warning(f"  Trial {issue['trial_number']}: {issue['description']}")
                if len(validation_issues) > 5:
                    logging.warning(f"  ... and {len(validation_issues) - 5} more issues")
            
            # Create participant audio
            looming_audio, tactile_audio, injection_log = self.create_participant_audio(
                participant_id, injection_map)
            
            # Verify synchronization
            verification = self.verify_audio_sync(injection_log)
            
            # Save audio files and validation report
            looming_path, tactile_path, log_path = self.save_audio_files(
                participant_id, looming_audio, tactile_audio, injection_log, validation_issues)
            
            return True, (looming_path, tactile_path, log_path)
            
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """
        Run audio generation for multiple participants.
        
        Args:
            participant_ids: List of participant IDs or None for all
            
        Returns:
            Dictionary with results
        """
        # If no participant IDs provided, find all design CSVs
        if participant_ids is None:
            # Find all participant design files
            design_dir = self.config['paths']['design_dir']
            design_files = [f for f in os.listdir(design_dir) if f.startswith('participant_') and f.endswith('_design.csv')]
            
            # Extract participant IDs
            participant_ids = []
            for filename in design_files:
                match = re.match(r'participant_(\d+)_design\.csv', filename)
                if match:
                    participant_ids.append(int(match.group(1)))
            
            participant_ids.sort()  # Sort numerically
        
        if self.config['debug_mode']:
            logging.info(f"=== Starting PPSAudioGenerator ===")
            logging.info(f"Processing {len(participant_ids)} participants: {participant_ids}")
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        # Summary
        if self.config['debug_mode']:
            success_count = sum(1 for info in results.values() if info['success'])
            logging.info(f"\n=== GENERATION SUMMARY ===")
            logging.info(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        
        return results

def main():
    """Main function to run the audio generator."""
    print("\n===== PPS EXPERIMENT AUDIO GENERATOR =====")
    print("This script generates participant-specific audio files with precise timing")
    print("It validates timestamp ordering: retentionslot < looming < tactile\n")
    
    # Configuration
    base_config = CONFIG.copy()
    
    # Auto-detect parent audio file (already implemented in the generator class)
    
    # By default, process all participant designs found in the directory
    participant_ids = None
    print("\nProcessing all available participants")
    
    # Create generator
    try:
        generator = PPSAudioGenerator(base_config)
        
        print(f"\nConfiguration:")
        print(f"  Parent audio: {generator.parent_audio_path}")
        print(f"  Design directory: {base_config['paths']['design_dir']}")
        print(f"  Output directory: {base_config['paths']['output_dir']}")
        print(f"  Timestamp hierarchy: retentionslot_timestamp < looming_stimulus_timestamp < tactile_stimulus_timestamp")
        
        # Run generator
        results = generator.run(participant_ids)
        
        # Print summary
        print("\n===== GENERATION SUMMARY =====")
        success_count = sum(1 for info in results.values() if info['success'])
        print(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        print("\nVALIDATION INFORMATION:")
        print("  - Only stimuli with correct timestamp ordering were included")
        print("  - Detailed reports saved as participant_*_validation_report.txt")
        print("  - Timestamps with validation issues were SKIPPED")
        
        for pid, info in results.items():
            if info['success']:
                looming_path, tactile_path, validation_path = info['paths']
                print(f"\nParticipant {pid}: Files generated")
                print(f"  Looming: {os.path.basename(looming_path)}")
                print(f"  Tactile: {os.path.basename(tactile_path)}")
                print(f"  Validation: {os.path.basename(validation_path)}")
            else:
                print(f"\nParticipant {pid}: ERROR - audio generation failed.")
                
    except Exception as e:
        print(f"Error: {e}")
        traceback.print_exc()
    
    print("\nAudio generation complete.")

if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (2334136846.py, line 95)